# PDC Exposure Explorer: a step-by-step notebook

Where and when did disasters happen, and how many people, households, schools and hospitals did PDC estimate to be exposed?
This notebook answers that from the **IFRC Montandon** STAC API (the `pdc-events`, `pdc-hazards` and `pdc-impacts` collections).
It shows every step: how the search is built, how the pages are retrieved, how the items become tables, and how the tables become figures and maps.
Cells that define functions start folded (a cell shows only its output; unfold it, or choose *Show code* in Colab, to read the function). The call that runs it is in the same cell.

> **Read this first.** PDC `affected_total` values are **PDC population or asset exposure estimates**.
> They are not deaths, injuries, displacement or confirmed humanitarian outcomes. Exposure is never added across events or countries.

**How to use it.** Run the cells from top to bottom (*Run → Run All Cells*). Nothing is requested from the API before step 5.
After you change the filters in step 3, run again **from step 4 down**. The pickers in steps 7 to 13 only redraw charts from the data
that is already loaded; they never call the API.

| Step | What happens |
|---|---|
| 1. Set up | Install (Colab only) and import |
| 2. Sign in | Your own Montandon token, kept in memory only |
| 3. Filters | Countries, years, months, hazards, measures |
| 4. Query | The search the filters turn into |
| 5. Retrieve | Month-by-month search, following every continuation link |
| 6. Process | Items become tables; snapshots become event families |
| 7. View | Choose the country and measure to look at |
| 8–14 | Overview, map, events, exposure, one event, country comparison, data quality |
| 15. Export | CSV, Excel, GeoJSON and a report |
| 16. Rules | What the numbers do and do not mean |

## 1. Set up

On **Colab** the first cell installs this project from GitHub (about a minute). On **Binder** and on your own computer it is already installed.
If Colab asks you to restart the session after the install, restart it and run the notebook again from the top.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # The repository must be public for this to work. It installs the project and its few dependencies.
    %pip install -q "git+https://github.com/arunissun/Gaurd-PDC-explorer-"

import base64
import hashlib
import json
import math
import re
import shutil
import threading
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta, timezone
from html import escape
from pathlib import Path
from urllib.parse import parse_qsl, unquote, urlencode, urljoin, urlparse, urlunparse

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
from openpyxl import Workbook, load_workbook
from plotly.subplots import make_subplots

# Safety plumbing and shared definitions come from the project package (explained just below).
from guard_pdc import __version__
from guard_pdc.api import ApiError, PdcApiProvider
from guard_pdc.config import MontandonConfig, find_token
from guard_pdc.countries import COUNTRY_NAMES, country_label, country_name
from guard_pdc.exports import public_frame, public_text, validate_export_bundle
from guard_pdc.maps import ALERT_AREA_NOTE, CLUSTER_NOTE, HAZARD_MAP_NOTES, MAP_NOTE, display_geometry, fetch_alert_areas
from guard_pdc.models import QuerySpec, canonical_payload_hash
from guard_pdc.report_charts import report_charts
from guard_pdc.taxonomy import (
    AGE_BAND_CATEGORIES, CAPITAL_UNIT_NOTE, DEFAULT_MEASURES, HAZARD_GROUP_ORDER, HAZARD_GROUPS, KNOWN_CATEGORIES,
    KNOWN_IMPACT_TYPES, MEASURE_CATEGORY, MEASURES, category_group, category_label, code_title_caveat, hazard_group,
    hazard_groups, measures_query, pdc_hazard_type,
)
from guard_pdc.theme import (
    ALERT_COLORS, CONTEXT, INK, INK_SECONDARY, PLOTLY_CONFIG, PRIMARY, SIZE_LEGEND_VALUES, SURFACE, apply_plotly_theme,
    compact, empty_figure, exposure_size, hazard_color, log_axis, sequential_scale, with_alpha,
)

### What stays in the project package, and why

Almost everything in this notebook is written out in the cells so that you can read it. A few things stay in the package (`src/guard_pdc/`) on purpose,
because they protect the token or define the shared vocabulary, and should not be copied around:

| Imported from | What it does |
|---|---|
| `guard_pdc.config` | Finds your token (environment variable or `.env`) and never prints it. `MontandonConfig` hides it from `print`. |
| `guard_pdc.api` | `PdcApiProvider.request_json` sends **one** read-only request: bounded retries of temporary errors, redirects refused, the token only ever sent to the Montandon address, and only `POST /search` and metadata `GET`s allowed. |
| `guard_pdc.countries`, `guard_pdc.taxonomy` | Country names, hazard groups, measure definitions (which PDC categories belong to *People*, *Schools* …). |
| `guard_pdc.theme` | The colours and the Plotly look shared with the dashboard. |
| `guard_pdc.maps.fetch_alert_areas` | Fetches and validates one event's PDC alert-area file: allow-listed host, no credentials, size cap, no redirects. |
| `guard_pdc.exports` | Removes credentials and local paths from exported text, and re-opens the export files to check them. |
| `guard_pdc.models` | `canonical_payload_hash`: a fingerprint of an item that ignores expiring signed-URL parameters. `QuerySpec`: the validated query, used only to describe the query in the exports. |
| `guard_pdc.report_charts` | The three small SVG charts of the exported report. |

## 2. Sign in

Montandon needs a personal API token. The next cell looks for it, in this order: **Colab Secrets** (a secret named `MONTANDON_API_TOKEN`),
the environment variable or a local `.env` file, and finally a hidden prompt.
The token is held in memory only. It is never written to a file, an export or this notebook, and the cells print only *where* it came from.

In [ ]:
token, token_source = None, None

if IN_COLAB:
    try:
        from google.colab import userdata

        token, token_source = userdata.get("MONTANDON_API_TOKEN"), "Colab Secrets"
    except Exception:
        pass  # no secret saved: try the next option

if not token:
    token, token_source = find_token()  # environment variable, then a .env file

if not token:
    import getpass

    token, token_source = getpass.getpass("Montandon API token (hidden; kept in memory only): "), "the prompt"

if not token or not token.strip():
    raise ValueError("No token was supplied. Create one in your Montandon account and run this cell again.")

config = MontandonConfig(api_token=token.strip())  # the token is hidden from print() and repr()
api = PdcApiProvider(config)                        # sends single read-only requests (see above)
del token

print(f"Signed in. The token came from {token_source}; it stays in memory only and is never written anywhere.")

## 3. Choose the filters

Pick **1 to 5 countries**, **up to 5 consecutive years**, the **months** in which events started, optional **hazards** (none selected means all),
and the **measures** to retrieve. *People* is always included. *Age groups* adds 14 age bands for every event: much more data, so only add it when you need it.

Each country is retrieved on its own, so every value belongs to the country it was retrieved for. A message under the widgets tells you if a choice is not allowed.
Years without PDC data (Montandon holds PDC events for 2015, 2016, 2020 and 2022 onwards, as checked on 2026-10-01) simply return no events.

In [ ]:
MAX_COUNTRIES, MAX_YEARS = 5, 5
MONTH_NAMES = ("Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec")
wide = {"description_width": "90px"}

countries_choice = widgets.SelectMultiple(
    options=[(country_label(code), code) for code in sorted(COUNTRY_NAMES, key=COUNTRY_NAMES.get)],
    value=("PHL",), rows=9, description="Countries", style=wide, layout=widgets.Layout(width="380px"),
)
years_choice = widgets.IntRangeSlider(
    value=(2024, 2024), min=2000, max=2026, description="Years", continuous_update=False, style=wide, layout=widgets.Layout(width="480px"),
)
month_buttons = [widgets.ToggleButton(value=True, description=name, layout=widgets.Layout(width="54px")) for name in MONTH_NAMES]
hazards_choice = widgets.SelectMultiple(
    options=[label for label, _ in HAZARD_GROUPS], value=(), rows=8, description="Hazards", style=wide, layout=widgets.Layout(width="380px"),
)
measure_boxes = {
    measure: widgets.Checkbox(value=measure in DEFAULT_MEASURES, description=MEASURES[measure][0], indent=False,
                              disabled=measure == "people", layout=widgets.Layout(width="200px"))
    for measure in (*DEFAULT_MEASURES, "age_groups")
}

In [ ]:
#@title Functions: read_filters
# `read_filters` turns the widget values into one plain dictionary and lists everything that is not allowed.
def read_filters():
    """The current choices, plus a list of problems that stop a retrieval."""
    countries = tuple(sorted(countries_choice.value, key=country_name))
    first, last = years_choice.value
    months = tuple(number for number, button in enumerate(month_buttons, start=1) if button.value)
    measures = tuple(name for name, box in measure_boxes.items() if box.value or name == "people")
    problems = []
    if not countries:
        problems.append("Choose at least one country.")
    if len(countries) > MAX_COUNTRIES:
        problems.append(f"Choose at most {MAX_COUNTRIES} countries ({len(countries)} are selected).")
    if last - first + 1 > MAX_YEARS:
        problems.append(f"Choose at most {MAX_YEARS} consecutive years ({first}–{last} is {last - first + 1}).")
    if not months:
        problems.append("Choose at least one month.")
    choice = {"countries": countries, "first_year": first, "last_year": last, "months": months,
              "hazards": tuple(hazards_choice.value), "measures": measures}
    return choice, problems

In [ ]:
#@title Functions: explain_filters
filter_message = widgets.HTML()


def explain_filters(_change=None):
    """Show what will be retrieved, or why the current choices are not allowed."""
    choice, problems = read_filters()
    if problems:
        filter_message.value = "<span style='color:#B8312F'><b>Not allowed yet:</b> " + " ".join(escape(p) for p in problems) + "</span>"
        return
    years = str(choice["first_year"]) if choice["first_year"] == choice["last_year"] else f"{choice['first_year']}–{choice['last_year']}"
    month_text = "all months" if len(choice["months"]) == 12 else ", ".join(MONTH_NAMES[m - 1] for m in choice["months"])
    partial_year = " <i>(2026 is a partial year.)</i>" if choice["last_year"] == 2026 else ""
    filter_message.value = (
        f"<b>Ready:</b> {escape(', '.join(country_name(c) for c in choice['countries']))} · {years} · {month_text} · "
        f"{escape(', '.join(choice['hazards']) or 'all hazards')} · {escape(', '.join(MEASURES[m][0] for m in choice['measures']))}.{partial_year}"
    )


for widget in (countries_choice, years_choice, hazards_choice, *month_buttons, *measure_boxes.values()):
    widget.observe(explain_filters, names="value")
explain_filters()

display(widgets.VBox([
    widgets.HBox([countries_choice, widgets.VBox([years_choice, widgets.HTML("<b>Months in which events started</b>"),
                                                  widgets.HBox(month_buttons, layout=widgets.Layout(flex_flow="row wrap", width="480px"))])]),
    widgets.HBox([hazards_choice, widgets.VBox([widgets.HTML("<b>Measures</b> (People is always included)"), *measure_boxes.values()])]),
    filter_message,
]))

## 4. Build the search

The filters become one **STAC search** per country, collection and month. A search is a JSON body sent with `POST /search`; its `filter` is written in CQL2:

* `datetime` is compared with an inclusive start and an exclusive end, so one month never overlaps the next. This is the **event start** month (a normal STAC time interval would also return events that merely overlap the month).
* `monty:country_codes` must **contain** the country (`a_contains`). Events listing several countries therefore appear in each of them.
* `monty:hazard_codes` must **overlap** the codes of the hazard groups you chose (`a_overlaps`); nothing is added when no hazard is chosen.
* For `pdc-impacts` only, `monty:impact_detail.type` and `.category` must equal the measures you chose. The retrieval never asks for all 19 categories unless you ask for them.

`fields` asks the server for only the properties used here, and `limit` is the size of a page.

In [ ]:
# First the filter choices are copied into plain settings that the rest of the notebook uses.
choice, problems = read_filters()
if problems:
    raise ValueError("Fix the filters in step 3 first: " + " ".join(problems))

COUNTRIES = choice["countries"]
YEARS = tuple(range(choice["first_year"], choice["last_year"] + 1))
REQUESTED_MONTHS = choice["months"]
WINDOWS = [(year, month) for year in YEARS for month in REQUESTED_MONTHS]  # one search window per month
HAZARD_CODES = tuple(code for label, codes in HAZARD_GROUPS if label in choice["hazards"] for code in codes)
IMPACT_TYPES, CATEGORIES = measures_query(choice["measures"])
HAS_AGE_BANDS = "age_groups" in choice["measures"]
# The five measures that get a peak and latest value per event (age bands are shown for one event at a time).
MEASURE_NAMES = [measure for measure in MEASURE_CATEGORY if MEASURE_CATEGORY[measure] in CATEGORIES]

PAGE_LIMIT = 250  # items per page; Montandon accepts at least 1,000, 250 keeps each response quick
COLLECTIONS = ("pdc-events", "pdc-hazards", "pdc-impacts")
FIELDS = [
    "id", "collection", "geometry", "bbox", "assets", "links",
    "properties.datetime", "properties.start_datetime", "properties.end_datetime", "properties.title",
    "properties.description", "properties.roles", "properties.monty:src_event_id", "properties.monty:corr_id",
    "properties.monty:episode_number", "properties.monty:country_codes", "properties.monty:hazard_codes",
    "properties.monty:hazard_detail", "properties.monty:impact_detail", "properties.processing:version",
]

In [ ]:
#@title Functions: equals, search_body, month_window, …
def equals(name, value):
    return {"op": "=", "args": [{"property": name}, value]}


def search_body(country, collection, start, end, limit=PAGE_LIMIT):
    """The POST /search body for one country, one collection and one date window (start inclusive, end exclusive)."""
    clauses = [
        {"op": ">=", "args": [{"property": "datetime"}, start]},
        {"op": "<", "args": [{"property": "datetime"}, end]},
        {"op": "a_contains", "args": [{"property": "monty:country_codes"}, country]},
    ]
    if HAZARD_CODES:
        clauses.append({"op": "a_overlaps", "args": [{"property": "monty:hazard_codes"}, list(HAZARD_CODES)]})
    if collection == "pdc-impacts":
        for name, values, every_value in (("monty:impact_detail.type", IMPACT_TYPES, KNOWN_IMPACT_TYPES),
                                          ("monty:impact_detail.category", CATEGORIES, KNOWN_CATEGORIES)):
            if not set(values) >= every_value:  # when everything is wanted, no clause is needed
                clauses.append(equals(name, values[0]) if len(values) == 1 else {"op": "or", "args": [equals(name, v) for v in values]})
    return {"collections": [collection], "limit": limit, "filter-lang": "cql2-json",
            "filter": {"op": "and", "args": clauses}, "fields": {"include": FIELDS}}


def month_window(year, month):
    """First instant of the month and of the next month (UTC)."""
    return datetime(year, month, 1, tzinfo=timezone.utc), datetime(year + month // 12, month % 12 + 1, 1, tzinfo=timezone.utc)


def iso(moment):
    return moment.strftime("%Y-%m-%dT%H:%M:%SZ")

In [ ]:
#@title Functions: query_fingerprint, describe_query
# Each country also gets a small description of its query and a fingerprint of it.
def query_fingerprint(query):
    return hashlib.sha256(json.dumps(query, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()


def describe_query(country):
    """The retrieval settings of one country as a plain dictionary, with its fingerprint."""
    query = {"analysis_mode": "country_detail", "country_code": country, "year": YEARS[0], "months": list(REQUESTED_MONTHS),
             "hazard_codes": list(HAZARD_CODES), "impact_types": list(IMPACT_TYPES), "categories": list(CATEGORIES), "source_mode": "api_only"}
    if len(YEARS) > 1:
        query["end_year"] = YEARS[-1]
    return {"query": query, "fingerprint": query_fingerprint(query)}


QUERIES = {country: describe_query(country) for country in COUNTRIES}
print(f"{len(COUNTRIES)} countr{'ies' if len(COUNTRIES) != 1 else 'y'} × {len(WINDOWS)} month window{'s' if len(WINDOWS) != 1 else ''}, retrieved one month at a time.")
print("Query fingerprints:", {country: info["fingerprint"][:12] for country, info in QUERIES.items()})


# One example of each kind of search body, for the first country and the first month, so you can see exactly what is sent:
first_start, first_end = (iso(moment) for moment in month_window(*WINDOWS[0]))
for collection in ("pdc-events", "pdc-impacts"):
    example = search_body(COUNTRIES[0], collection, first_start, first_end)
    example["fields"] = {"include": [f"… {len(FIELDS)} field names …"]}
    print(f"POST {config.url('search')}   [{collection}]")
    print(json.dumps(example, indent=1), "\n")

## 5. Retrieve

**This is the first step that talks to the API.** For every country the notebook searches `pdc-events` month by month, then searches `pdc-hazards` and `pdc-impacts` only for the months that returned events
(hazard and impact items carry their event's date, so other months cannot hold any).

How one window is read, in plain steps:

1. `POST /search` with the body from step 4.
2. Read the `features`. If the response has a `next` link, send **that** request (a `POST` with its own body) and keep going. The API never reports a total (`numberMatched` is intentionally absent), so the only proof that a window is complete is a last page **without** a `next` link.
3. If a window holds 25,000 items and has more pages, it is **split** into weeks, then days. If a request keeps failing after the bounded retries, the window is split in half or asked for in smaller pages. Nothing is sampled or cut short.
4. A window that still cannot be read is reported as **incomplete**, never hidden.

Authentication errors stop everything at once. Each cell below defines its functions and, where there is something to run, runs them.

In [ ]:
#@title Functions: request_page, next_request
SEARCH_URL = config.url("search")
MIN_PAGE_LIMIT = 50                      # smallest page size tried after a failed request
ITEM_CAP = 25_000                        # unique items one window may hold before it is split into weeks, then days
FAILURE_BUDGET = 150                     # failed requests tolerated in one retrieval before windows are reported instead of retried
OVERLOADED = {429, 500, 502, 503, 504}   # statuses that mean "the server is struggling"
ONE_DAY = timedelta(days=1)
budget = {"left": FAILURE_BUDGET, "lock": threading.Lock()}


class WindowTooLarge(Exception):
    """A window reached ITEM_CAP and still has pages left."""


class IncompletePagination(ApiError):
    """The continuation links could not be followed to the end, so the window cannot be called complete."""


def request_page(url, method, body, stats, page_number):
    """Send one read-only request; if it fails, say which page it was."""
    try:
        return api.request_json(url, method=method, body=body, stats=stats)
    except ApiError as error:
        error.page_number = page_number  # fetch_window uses this to decide how to retry
        raise


def next_request(url, link, previous_body):
    """The (url, method, body) that a `next` link asks for."""
    method = str(link.get("method") or "GET").upper()
    if method != "POST":
        return urljoin(url, link["href"]), method, None
    if not link.get("body"):
        raise IncompletePagination("a POST continuation link has no body")
    return urljoin(url, link["href"]), method, ({**previous_body, **link["body"]} if link.get("merge") else link["body"])

In [ ]:
#@title Functions: read_pages
def read_pages(body, stats):
    """POST one search body and follow every continuation link; returns (items by ID with their page, items returned, items per page)."""
    url, method, request_body = SEARCH_URL, "POST", body
    found, per_page, seen = {}, [], set()
    while True:
        request_key = (method, url, json.dumps(request_body, sort_keys=True))
        if request_key in seen:
            raise IncompletePagination("the API returned the same continuation link twice")
        seen.add(request_key)
        page = request_page(url, method, request_body, stats, len(per_page) + 1)
        if not isinstance(page.get("features"), list):
            raise IncompletePagination("the API response has no list of features")
        per_page.append(len(page["features"]))
        for item in page["features"]:
            found.setdefault(item["id"], (item, len(per_page)))
        link = next((link for link in page.get("links") or [] if link.get("rel") == "next"), None)
        if link is None:
            return found, sum(per_page), per_page  # no continuation link: the window is complete
        if len(found) >= ITEM_CAP:
            raise WindowTooLarge
        url, method, request_body = next_request(url, link, request_body)

In [ ]:
#@title Functions: window_record, spend_failure
# A *job* describes a window (country, collection, start, end, page size).
def window_record(job, *, found=None, returned=0, pages=(), failure=None, status=None):
    """One line of the retrieval log (plus the items, which are moved to RAW afterwards)."""
    found = found or {}
    return {"country": job["country"], "collection": job["collection"], "start": iso(job["start"]), "end": iso(job["end"]), "limit": job["limit"],
            "pages": len(pages), "items": len(found), "returned": returned, "duplicates": returned - len(found), "retries": 0,
            "complete": failure is None, "note": ", ".join(job["notes"]), "failure": failure, "status": status,
            "retrieved_at": iso(datetime.now(timezone.utc)), "entries": found}


def spend_failure():
    """Count one failed request against the budget; False once it is used up."""
    with budget["lock"]:
        budget["left"] -= 1
        return budget["left"] >= 0

In [ ]:
#@title Functions: child, fetch_window
def child(job, start, end, note, limit=None):
    """The same job for a smaller window (or smaller pages), remembering why it was split."""
    return {**job, "start": start, "end": end, "limit": limit or job["limit"], "notes": job["notes"] + (note,)}


def fetch_window(job, stats):
    """Read one date window. A window that is too large or keeps failing is split; one that cannot be read is reported."""
    body = search_body(job["country"], job["collection"], iso(job["start"]), iso(job["end"]), job["limit"])
    try:
        found, returned, pages = read_pages(body, stats)
    except IncompletePagination as error:  # reported, never treated as a finished window
        return [window_record(job, failure=f"incomplete pagination: {error}")]
    except WindowTooLarge:
        return split_by_size(job, stats)
    except ApiError as error:
        return handle_api_error(job, stats, error)
    return [window_record(job, found=found, returned=returned, pages=pages)]

In [ ]:
#@title Functions: split_by_size, handle_api_error
def split_by_size(job, stats):
    """A window with 25,000 items and more pages is read again as weeks, then days. One day over the cap needs narrower filters."""
    start, end = job["start"], job["end"]
    if end - start <= ONE_DAY:
        raise ValueError(f"One day of {job['collection']} for {country_name(job['country'])} holds more than {ITEM_CAP:,} items. "
                         "Narrow the filters (fewer hazards or measures) and try again.")
    step = timedelta(days=7) if end - start > timedelta(days=7) else ONE_DAY
    parts = [(start + n * step, min(start + (n + 1) * step, end)) for n in range(math.ceil((end - start) / step))]
    return [record for a, b in parts for record in fetch_window(child(job, a, b, "split: item cap"), stats)]


def handle_api_error(job, stats, error):
    """Temporary failures: ask for each half of the window, then for smaller pages. Anything else stops the retrieval."""
    if not error.recoverable:
        raise error  # authentication, forbidden, malformed answer: stop everything
    if not spend_failure():
        return [window_record(job, failure=f"{error} (failure budget spent: not retried)", status=error.status)]
    start, end = job["start"], job["end"]
    if (error.page_number > 1 or error.status is None or error.status in OVERLOADED) and end - start > ONE_DAY:
        middle = (start + (end - start) / 2).replace(microsecond=0)
        return fetch_window(child(job, start, middle, "split: API error"), stats) + fetch_window(child(job, middle, end, "split: API error"), stats)
    if job["limit"] > MIN_PAGE_LIMIT:
        return fetch_window(child(job, start, end, "smaller pages", limit=100 if job["limit"] > 100 else MIN_PAGE_LIMIT), stats)
    return [window_record(job, failure=str(error), status=error.status)]

In [ ]:
#@title Functions: fetch_month, months_of
def fetch_month(country, collection, year, month):
    """All records of one month; the retries counted along the way are shown on the first record."""
    start, end = month_window(year, month)
    stats = {"retries": 0}
    records = fetch_window({"country": country, "collection": collection, "start": start, "end": end, "limit": PAGE_LIMIT, "notes": ()}, stats)
    records[0]["retries"] = stats["retries"]
    return records


def months_of(records, country):
    """Months that returned events, plus months whose events could not be read (they may hold some)."""
    months = set()
    for record in records:
        if record["country"] != country:
            continue
        if not record["complete"]:
            months.add((int(record["start"][:4]), int(record["start"][5:7])))
        for item, _ in record["entries"].values():
            when = (item.get("properties") or {}).get("datetime")
            if isinstance(when, str) and len(when) >= 7 and when[4] == "-":
                months.add((int(when[:4]), int(when[5:7])))
    return months & set(WINDOWS)

In [ ]:
#@title Functions: run
def run(jobs, label):
    """Run (country, collection, year, month) jobs on a few threads and show progress; returns every window record."""
    bar = widgets.IntProgress(max=max(len(jobs), 1), description=label, bar_style="info", layout=widgets.Layout(width="520px"))
    line = widgets.HTML()
    display(widgets.HBox([bar, line]))
    records = []
    with ThreadPoolExecutor(max_workers=config.max_workers) as pool:
        futures = {pool.submit(fetch_month, *job): job for job in jobs}
        try:
            for future in as_completed(futures):
                records.extend(future.result())  # raises at once on an authentication error
                country, _, year, month = futures[future]
                bar.value += 1
                line.value = f"{escape(country_name(country))} · {year}-{month:02d} · {sum(r['items'] for r in records):,} items so far"
        except BaseException as error:
            if isinstance(error, ApiError) and error.status in (401, 403):
                print("Montandon refused the token. Check it in step 2 (run that cell again) and make sure it is still valid.")
            pool.shutdown(wait=False, cancel_futures=True)  # do not send the windows still waiting
            raise
    bar.bar_style = "success" if all(record["complete"] for record in records) else "warning"
    return records


started = time.perf_counter()
budget["left"] = FAILURE_BUDGET
records = run([(country, "pdc-events", year, month) for country in COUNTRIES for year, month in WINDOWS], "Events")


# Phase 2: hazards and exposure, only for the months that returned events.
follow_up = [(country, collection, year, month) for country in COUNTRIES for year, month in sorted(months_of(records, country))
             for collection in ("pdc-hazards", "pdc-impacts")]
records += run(follow_up, "Hazards & exposure")

In [ ]:
#@title Functions: store_items
def store_items(records):
    """Items by country and collection (the first copy wins), each with where it came from. Repeated item IDs are counted per country, and so are repeats whose content differs."""
    raw = {country: {collection: {} for collection in COLLECTIONS} for country in COUNTRIES}
    repeated, conflicts = defaultdict(int), defaultdict(int)
    for record in sorted(records, key=lambda r: (r["country"], r["collection"], r["start"])):
        store = raw[record["country"]][record["collection"]]
        for item_id, (item, page) in record["entries"].items():
            if item_id not in store:
                store[item_id] = {"item": item, "window": f"{record['start']}/{record['end']}", "page": page, "retrieved_at": record["retrieved_at"]}
                continue
            repeated[record["country"]] += 1
            conflicts[record["country"]] += canonical_payload_hash(item) != canonical_payload_hash(store[item_id]["item"])
    return raw, repeated, conflicts


RAW, REPEATED_ITEM_IDS, PAYLOAD_CONFLICTS = store_items(records)
RETRIEVAL_LOG = pd.DataFrame([{key: value for key, value in record.items() if key != "entries"} for record in records])
RETRIEVAL_LOG["month"] = RETRIEVAL_LOG["start"].str[:7]
RETRIEVED_AT = iso(datetime.now(timezone.utc))
RETRIEVAL_SECONDS = time.perf_counter() - started
INCOMPLETE = RETRIEVAL_LOG[~RETRIEVAL_LOG["complete"]]


unique_items = sum(len(store) for country in RAW.values() for store in country.values())
print(f"Retrieved in {RETRIEVAL_SECONDS:.0f} s: {RETRIEVAL_LOG['pages'].sum():,} pages, {unique_items:,} unique items"
      f" ({RETRIEVAL_LOG['retries'].sum()} retried requests, {RETRIEVAL_LOG['note'].ne('').sum()} windows split, {sum(REPEATED_ITEM_IDS.values())} repeated item IDs).")
if len(INCOMPLETE):
    print(f"\nWARNING: {len(INCOMPLETE)} window(s) could not be read, so the result is incomplete. Run this step again to retry:")
    for row in INCOMPLETE.itertuples():
        print(f"  {country_name(row.country)} · {row.collection} · {row.start[:10]} to {row.end[:10]}: {public_text(str(row.failure))}")
else:
    print("Every window was read to its last page: the result is complete.")


# Items per country and month: the retrieval record.
per_month = RETRIEVAL_LOG.pivot_table(index=["country", "month"], columns="collection", values="items", aggfunc="sum", fill_value=0)
per_month = per_month.reindex(columns=list(COLLECTIONS), fill_value=0).rename(index=country_name, level="country")
with pd.option_context("display.max_rows", 80):
    display(per_month)

## 6. Process: from items to tables

The API answers with STAC items (nested JSON). This step turns them into flat tables and keeps **every original value**: the value as published, its unit (`None` when PDC gives none),
category, type, estimate type, identifiers, geometry, dates, links, processing version and where each item came from (provenance).
Three item tables come out: **events** (one row per event item, called a *snapshot*), **hazards** (the PDC alert level and hazard details) and **impacts** (one row per exposure value).
Then the snapshots are grouped into **event families** and summarised **per event**.

**Event families.** PDC publishes an event many times as it is updated, so one real event has several snapshots. `monty:src_event_id` identifies the family; repeating it is *not* a duplicate.
Only an item **without** a source event ID falls back to its unchanged `monty:corr_id`, and that fallback is labelled `legacy_corr_id_fallback`.
A **repeated item ID**, by contrast, would be a duplicate signal and is counted.

**Exposure time.** An event's `datetime` stays constant across its snapshots. The time of each exposure estimate is encoded in the item ID by PDC's transformer; it is read from there (only for known processing versions) and labelled as such. It is *not* an observed impact-report time.

In [ ]:
#@title Functions: exposure_snapshot_time
UUID = r"[0-9a-fA-F]{8}-(?:[0-9a-fA-F]{4}-){3}[0-9a-fA-F]{12}"
EVENT_OR_HAZARD_ID = re.compile(rf"^pdc-(?:event|hazard)-{UUID}-[A-Za-z0-9_.:~-]+-(?P<epoch>\d{{10,13}})$")
IMPACT_ID = re.compile(rf"^pdc-impact-{UUID}-[A-Za-z0-9_.:~-]+-(?P<epoch>\d{{10,13}})-\d+-(?:population|capital)-[A-Za-z0-9_]+-[A-Za-z]{{3}}$")
KNOWN_VERSIONS = {"0.1.1", *(f"0.2.{patch}" for patch in range(8))}  # processing versions whose ID layout was checked
SNAPSHOT_METHOD = "pdc_item_id_exposure_timestamp"
SENSITIVE_QUERY = ("token", "secret", "signature", "credential", "accesskey", "authorization", "expires")
CURRENCY_QUOTE = re.compile(r"\$\s?[\d.,]+\s*(?:Thousand|Million|Billion|Trillion)?\s*\(USD\)\s*of infrastructure", re.IGNORECASE)


def exposure_snapshot_time(item_id, processing_version=None):
    """The exposure timestamp PDC's transformer writes into the item ID, or None when the ID or version is not one we checked."""
    if processing_version is not None:
        if not isinstance(processing_version, str) or processing_version.removeprefix("v") not in KNOWN_VERSIONS:
            return None
    match = (IMPACT_ID if item_id.startswith("pdc-impact-") else EVENT_OR_HAZARD_ID).fullmatch(item_id)
    if match is None:
        return None
    seconds = int(match["epoch"])
    if seconds >= 1_000_000_000_000:  # milliseconds
        seconds //= 1000
    try:
        moment = datetime.fromtimestamp(seconds, timezone.utc)
    except (OverflowError, OSError, ValueError):
        return None
    return moment.isoformat().replace("+00:00", "Z") if 2000 <= moment.year <= 2100 else None

In [ ]:
#@title Functions: family_key, point_parts
def family_key(properties, item_id):
    """The event-family key: the PDC source event ID; without one, the unchanged corr_id (labelled legacy); without either, the item itself."""
    source = properties.get("monty:src_event_id")
    if source:
        return str(source)
    corr_id = properties.get("monty:corr_id")
    return f"legacy:{corr_id}" if corr_id else f"item:{item_id}"


def point_parts(geometry):
    """(type, longitude, latitude, valid): coordinates are only filled in for a valid Point."""
    if not isinstance(geometry, dict):
        return None, None, None, False
    kind, coordinates = geometry.get("type"), geometry.get("coordinates")
    if kind != "Point" or not isinstance(coordinates, list) or len(coordinates) < 2:
        return (str(kind) if kind else None), None, None, False
    longitude, latitude = coordinates[:2]
    valid = (all(isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value) for value in (longitude, latitude))
             and -180 <= longitude <= 180 and -90 <= latitude <= 90)
    return "Point", (float(longitude) if valid else None), (float(latitude) if valid else None), valid

In [ ]:
#@title Functions: asset_href, footprint_url, report_url, …
def asset_href(assets, name):
    asset = assets.get(name) if isinstance(assets, dict) else None
    return asset.get("href") if isinstance(asset, dict) and isinstance(asset.get("href"), str) else None


def footprint_url(assets):
    """The unsigned object URL of the PDC 'Maps' asset. The signing parameters expire and are dropped; the object itself is public."""
    href = asset_href(assets, "Maps")
    parsed = urlparse(href) if href else None
    return urlunparse(parsed._replace(query="", fragment="")) if parsed and parsed.scheme == "https" and parsed.hostname else None


def report_url(assets):
    """The PDC HazardBrief link, without any credential-like query values."""
    href = asset_href(assets, "report")
    parsed = urlparse(href) if href else None
    if not parsed or parsed.scheme != "https":
        return None
    query = [(key, value) for key, value in parse_qsl(parsed.query) if not any(part in key.lower() for part in SENSITIVE_QUERY)]
    return urlunparse(parsed._replace(query=urlencode(query), fragment=""))


def currency_quote(description):
    """PDC's own wording about capital exposure, quoted verbatim (the structured data carries no currency)."""
    match = CURRENCY_QUOTE.search(description) if isinstance(description, str) else None
    return " ".join(match.group(0).split()) if match else None

In [ ]:
#@title Functions: related_targets
def related_targets(item):
    """(collection, item ID, href) of every rel=related link of an item."""
    found = []
    for link in item.get("links") or []:
        href = link.get("href")
        if link.get("rel") != "related" or not isinstance(href, str) or not href:
            continue
        parts = [unquote(part) for part in urlparse(href).path.split("/") if part and part != ".."]
        at = parts.index("collections") if "collections" in parts else None
        if at is not None and len(parts) > at + 3 and parts[at + 2] == "items":
            found.append((parts[at + 1], parts[at + 3].removesuffix(".json"), href))
        else:
            collection = link.get("collection") if isinstance(link.get("collection"), str) else next((p for p in parts if p.startswith("pdc-")), None)
            found.append((collection, parts[-1].removesuffix(".json") if parts else None, href))
    return found

In [ ]:
#@title Functions: provenance, source_id_of
PROVENANCE_COLUMNS = ["provider", "availability", "retrieval_window", "search_page", "retrieved_at", "payload_hash", "query_fingerprint"]
GEOMETRY_COLUMNS = ["geometry", "geometry_type", "longitude", "latitude", "valid_geometry"]
EVENT_COLUMNS = [
    "family_key", "event_title", "src_event_id", "corr_id", "event_item_id", "event_datetime", "start_datetime", "end_datetime",
    "snapshot_time", "snapshot_time_method", "processing_version", "episode_number", "hazard_codes", "country_codes",
    "linked_hazard_ids", "linked_impact_ids", "footprint_url", "report_url", "description_currency_quote",
    *GEOMETRY_COLUMNS, *PROVENANCE_COLUMNS, "correlation_status",
]
HAZARD_COLUMNS = [
    "family_key", "src_event_id", "corr_id", "event_item_id", "hazard_item_id", "item_datetime", "start_datetime", "end_datetime",
    "snapshot_time", "snapshot_time_method", "episode_number", "country_codes", "hazard_codes", "hazard_labels",
    "severity_value", "severity_unit", "severity_label", "estimate_type", *GEOMETRY_COLUMNS, *PROVENANCE_COLUMNS,
    "correlation_method", "correlation_status",
]
IMPACT_COLUMNS = [
    "family_key", "src_event_id", "corr_id", "event_item_id", "impact_item_id", "country_code", "impact_type", "impact_category",
    "category_label", "category_group", "original_value", "numeric_value", "original_unit", "estimate_type", "item_datetime",
    "detail_position", "snapshot_time", "snapshot_time_method", *GEOMETRY_COLUMNS, "processing_version", *PROVENANCE_COLUMNS,
    "correlation_method", "correlation_status",
]


def provenance(item, source, country):
    """Where an item came from: provider, search window and page, retrieval time, a payload fingerprint and the query fingerprint."""
    return {"provider": "api", "availability": "api_only", "retrieval_window": source["window"], "search_page": source["page"],
            "retrieved_at": source["retrieved_at"], "payload_hash": canonical_payload_hash(item), "query_fingerprint": QUERIES[country]["fingerprint"]}


def source_id_of(properties):
    source = properties.get("monty:src_event_id")
    return str(source) if source is not None else None

In [ ]:
#@title Functions: tidy_event
def tidy_event(item, source, country):
    """One row per event item (a snapshot of an event)."""
    p = item.get("properties") or {}
    kind, longitude, latitude, valid = point_parts(item.get("geometry"))
    snapshot = exposure_snapshot_time(item["id"], p.get("processing:version"))
    links = related_targets(item)
    return {
        "family_key": family_key(p, item["id"]), "event_title": p.get("title"), "src_event_id": source_id_of(p), "corr_id": p.get("monty:corr_id"),
        "event_item_id": item["id"], "event_datetime": p.get("datetime"), "start_datetime": p.get("start_datetime"), "end_datetime": p.get("end_datetime"),
        "snapshot_time": snapshot, "snapshot_time_method": SNAPSHOT_METHOD if snapshot else None,
        "processing_version": p.get("processing:version"), "episode_number": p.get("monty:episode_number"),
        "hazard_codes": tuple(p.get("monty:hazard_codes") or ()), "country_codes": tuple(p.get("monty:country_codes") or ()),
        "linked_hazard_ids": tuple(dict.fromkeys(i for c, i, _ in links if c == "pdc-hazards" and i)),
        "linked_impact_ids": tuple(dict.fromkeys(i for c, i, _ in links if c == "pdc-impacts" and i)),
        "footprint_url": footprint_url(item.get("assets")), "report_url": report_url(item.get("assets")),
        "description_currency_quote": currency_quote(p.get("description")),
        "geometry": item.get("geometry"), "geometry_type": kind, "longitude": longitude, "latitude": latitude, "valid_geometry": valid,
        **provenance(item, source, country),
    }

In [ ]:
#@title Functions: tidy_hazard
def tidy_hazard(item, source, country):
    """One row per hazard item: the PDC alert level (severity label) and hazard details."""
    p = item.get("properties") or {}
    detail = p.get("monty:hazard_detail") if isinstance(p.get("monty:hazard_detail"), dict) else {}
    kind, longitude, latitude, valid = point_parts(item.get("geometry"))
    snapshot = exposure_snapshot_time(item["id"], p.get("processing:version"))
    codes = tuple(p.get("monty:hazard_codes") or ())
    return {
        "family_key": family_key(p, item["id"]), "src_event_id": source_id_of(p), "corr_id": p.get("monty:corr_id"), "event_item_id": None,
        "hazard_item_id": item["id"], "item_datetime": p.get("datetime"), "start_datetime": p.get("start_datetime"), "end_datetime": p.get("end_datetime"),
        "snapshot_time": snapshot, "snapshot_time_method": SNAPSHOT_METHOD if snapshot else None, "episode_number": p.get("monty:episode_number"),
        "country_codes": tuple(p.get("monty:country_codes") or ()), "hazard_codes": codes, "hazard_labels": hazard_groups(codes),
        "severity_value": detail.get("severity_value"), "severity_unit": detail.get("severity_unit"),
        "severity_label": detail.get("severity_label"), "estimate_type": detail.get("estimate_type"),
        "geometry": item.get("geometry"), "geometry_type": kind, "longitude": longitude, "latitude": latitude, "valid_geometry": valid,
        **provenance(item, source, country),
    }

In [ ]:
#@title Functions: tidy_impacts
def tidy_impacts(item, source, country):
    """One row per exposure value (an impact item can carry several details). The value is kept exactly as published."""
    p = item.get("properties") or {}
    details = p.get("monty:impact_detail")
    details = [details] if isinstance(details, dict) else [d for d in details if isinstance(d, dict)] if isinstance(details, list) else []
    countries = tuple(p.get("monty:country_codes") or ())
    kind, longitude, latitude, valid = point_parts(item.get("geometry"))
    snapshot = exposure_snapshot_time(item["id"], p.get("processing:version"))
    rows = []
    for position, detail in enumerate(details):
        value, category = detail.get("value"), str(detail.get("category") or "")
        rows.append({
            "family_key": family_key(p, item["id"]), "src_event_id": source_id_of(p), "corr_id": p.get("monty:corr_id"), "event_item_id": None,
            "impact_item_id": item["id"], "country_code": countries[0] if len(countries) == 1 else None,
            "impact_type": str(detail.get("type") or ""), "impact_category": category, "category_label": category_label(category), "category_group": category_group(category),
            "original_value": value, "numeric_value": value if isinstance(value, (int, float)) and not isinstance(value, bool) else None,
            "original_unit": detail.get("unit"), "estimate_type": detail.get("estimate_type"), "item_datetime": p.get("datetime"), "detail_position": position,
            "snapshot_time": snapshot, "snapshot_time_method": SNAPSHOT_METHOD if snapshot else None,
            "geometry": item.get("geometry"), "geometry_type": kind, "longitude": longitude, "latitude": latitude, "valid_geometry": valid,
            "processing_version": p.get("processing:version"), **provenance(item, source, country),
        })
    return rows

### Checking the links between events, hazards and exposure

Every PDC item carries `rel="related"` links to its event, hazard and exposure items. They are used to **locate and validate** the exact snapshot an exposure value belongs to:
the linked item must exist in the result, must carry the same source event ID, and its episode and time span must agree.
An exposure item that no link validates is matched by its source event ID instead; if several snapshots could own it, the edge is kept as **ambiguous**; it is never assigned to "the first one".
Items without a source event ID are matched by the unchanged `corr_id` and labelled `legacy_corr_id_fallback`, with the same checks plus country and hazard.
Nothing here ties items from *different* sources together; correlation IDs alone never do that.

In [ ]:
#@title Functions: parse_time, times_overlap, same_episode, …
def parse_time(value):
    if not isinstance(value, str) or not value:
        return None
    try:
        moment = datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        return None
    return moment if moment.tzinfo else moment.replace(tzinfo=timezone.utc)


def times_overlap(first, second):
    """Do the two items' time spans overlap? None when a time is missing."""
    spans = [(parse_time(p.get("start_datetime") or p.get("datetime")), parse_time(p.get("end_datetime") or p.get("datetime"))) for p in (first, second)]
    if any(moment is None for span in spans for moment in span):
        return None
    return max(spans[0][0], spans[1][0]) <= min(spans[0][1], spans[1][1])


def same_episode(first, second):
    left, right = first.get("monty:episode_number"), second.get("monty:episode_number")
    return None if left is None or right is None else left == right


def fallback_compatible(first, second):
    """Extra checks for the corr_id fallback: shared country and hazard, compatible episode and time."""
    for name in ("monty:country_codes", "monty:hazard_codes"):
        left, right = set(first.get(name) or ()), set(second.get(name) or ())
        if left and right and not left & right:
            return False
    return same_episode(first, second) is not False and times_overlap(first, second) is not False

In [ ]:
#@title Functions: judge, edge_row
RANK = {"validated": 0, "missing_companion": 1, "ambiguous": 2, "conflict": 3}  # how serious a status is
EDGE_COLUMNS = ["src_event_id", "event_item_id", "target_collection", "target_item_id", "target_src_event_id", "related_href", "related_role",
                "link_method", "src_event_id_match", "episode_match", "time_match", "correlation_fallback", "status", "candidate_count", "review_reason"]


def judge(ep, target, method, candidates):
    """The status of one event-to-item edge, and the reason when it is not 'validated'."""
    tp = (target or {}).get("properties") or {}
    source, target_source = source_id_of(ep), source_id_of(tp)
    if target is None:
        return "missing_companion", "related item was not present in the bounded result"
    if source and target_source and source != target_source:
        return "conflict", "related items have different source event IDs"
    if method == "legacy_corr_id_fallback" and (not ep.get("monty:corr_id") or ep.get("monty:corr_id") != tp.get("monty:corr_id") or not fallback_compatible(ep, tp)):
        return "conflict", "legacy correlation ID or compatibility checks failed"
    if candidates > 1:
        return "ambiguous", "multiple candidate records remain"
    if same_episode(ep, tp) is False or times_overlap(ep, tp) is False:
        return "conflict", "episode or time compatibility failed"
    return "validated", None


def edge_row(event, target, target_collection, target_id, href, method, candidates):
    """One event-to-item edge with its validation result."""
    ep, tp = event.get("properties") or {}, (target or {}).get("properties") or {}
    status, reason = judge(ep, target, method, candidates)
    source, target_source = source_id_of(ep), source_id_of(tp)
    return {
        "src_event_id": source, "event_item_id": event["id"], "target_collection": target_collection, "target_item_id": target_id,
        "target_src_event_id": target_source, "related_href": href, "related_role": "related" if href else None, "link_method": method,
        "src_event_id_match": source == target_source if source is not None and target_source is not None else None,
        "episode_match": same_episode(ep, tp) if target else None, "time_match": times_overlap(ep, tp) if target else None,
        "correlation_fallback": method == "legacy_corr_id_fallback", "status": status, "candidate_count": candidates, "review_reason": reason,
    }

In [ ]:
#@title Functions: edges_from_links
def edges_from_links(store):
    """Edges from the rel=related links, read from both sides. Also returns the (event, collection, item) pairs a link covered."""
    edges, linked = [], set()
    for collection in COLLECTIONS:
        for item in store[collection].values():
            for target_collection, target_id, href in related_targets(item):
                if not target_id or target_collection not in COLLECTIONS:
                    continue
                if collection == "pdc-events" and target_collection != "pdc-events":
                    event, target = item, store[target_collection].get(target_id)
                elif collection != "pdc-events" and target_collection == "pdc-events":
                    event, target, target_collection, target_id = store["pdc-events"].get(target_id), item, collection, item["id"]
                    if event is None:
                        continue
                else:
                    continue
                linked.add((event["id"], target_collection, target_id))
                if target is None:
                    if target_collection == "pdc-hazards":  # exposure items outside the chosen measures are not expected
                        edges.append(edge_row(event, None, target_collection, target_id, href, "related_link", 0))
                    continue
                has_ids = source_id_of(event.get("properties") or {}) and source_id_of(target.get("properties") or {})
                edges.append(edge_row(event, target, target_collection, target_id, href, "related_link" if has_ids else "legacy_corr_id_fallback", 1))
    return edges, linked

In [ ]:
#@title Functions: candidates_for, edges_from_source_ids
def candidates_for(target, by_source, by_corr):
    """The events a hazard or exposure item could belong to, and how they were found."""
    properties = target.get("properties") or {}
    if source_id_of(properties) and by_source.get(source_id_of(properties)):
        return by_source[source_id_of(properties)], "source_event_id"
    corr_id = properties.get("monty:corr_id")
    if not isinstance(corr_id, str) or not corr_id:
        return [], "source_event_id"
    return [event for event in by_corr.get(corr_id, []) if fallback_compatible(event.get("properties") or {}, properties)], "legacy_corr_id_fallback"


def edges_from_source_ids(store, edges, linked):
    """Edges for hazard and exposure items that no link validated: match on the source event ID, else on the unchanged corr_id."""
    validated = {(e["target_collection"], e["target_item_id"]) for e in edges if e["link_method"] == "related_link" and e["status"] == "validated"}
    by_source, by_corr = defaultdict(list), defaultdict(list)
    for event in store["pdc-events"].values():
        properties = event.get("properties") or {}
        if source_id_of(properties):
            by_source[source_id_of(properties)].append(event)
        if properties.get("monty:corr_id"):
            by_corr[properties["monty:corr_id"]].append(event)
    found = []
    for target_collection in ("pdc-hazards", "pdc-impacts"):
        for target in store[target_collection].values():
            if (target_collection, target["id"]) not in validated:
                candidates, method = candidates_for(target, by_source, by_corr)
                found += [edge_row(event, target, target_collection, target["id"], None, method, len(candidates))
                          for event in candidates if (event["id"], target_collection, target["id"]) not in linked]
    return found

In [ ]:
#@title Functions: strongest_edges, link_evidence, attach_links
def strongest_edges(edges):
    """One row per event, item and method: keep the strongest result (validated, then missing, ambiguous, conflict)."""
    grouped = defaultdict(list)
    for edge in edges:
        grouped[(edge["event_item_id"], edge["target_collection"], edge["target_item_id"], edge["link_method"])].append(edge)
    rows = []
    for key in sorted(grouped):
        values = grouped[key]
        strongest = max(values, key=lambda edge: RANK.get(edge["status"], 4))
        reasons = list(dict.fromkeys(edge["review_reason"] for edge in values if edge["review_reason"]))
        rows.append({**strongest, "candidate_count": max(edge["candidate_count"] for edge in values), "review_reason": "; ".join(reasons) or None})
    return pd.DataFrame(rows, columns=EDGE_COLUMNS)


def link_evidence(country):
    """One row per event-to-hazard or event-to-impact edge: the related links first, then the source event ID."""
    store = {collection: {i: v["item"] for i, v in RAW[country][collection].items()} for collection in COLLECTIONS}
    edges, linked = edges_from_links(store)
    return strongest_edges(edges + edges_from_source_ids(store, edges, linked))


def attach_links(table, id_column, edges, collection):
    """Give each hazard or impact row its validated event snapshot and the method and status of its strongest edge."""
    mine = edges[edges["target_collection"] == collection]
    strongest = {}
    for row in mine.itertuples():
        best = strongest.get(row.target_item_id)
        if best is None or RANK.get(row.status, 4) > RANK.get(best.status, 4):
            strongest[row.target_item_id] = row
    validated = mine[mine["status"] == "validated"].groupby("target_item_id")["event_item_id"].agg(lambda ids: set(ids))
    table["event_item_id"] = table[id_column].map(lambda i: next(iter(validated[i])) if i in validated.index and len(validated[i]) == 1 else None)
    table["correlation_method"] = table[id_column].map(lambda i: strongest[i].link_method if i in strongest else None)
    table["correlation_status"] = table[id_column].map(lambda i: strongest[i].status if i in strongest else "unlinked")
    return table

### Event families and the per-event summary

**Families.** All snapshots with the same family key become one row: title, first and last date, countries, hazard codes, number of snapshots and of exposure items, and how the family was identified.

**The per-event summary** has one row per family and is what the charts and the map use. For each measure (People, Households, Schools, Hospitals, Capital exposure) it keeps:

* the **peak**: the largest value among the event's snapshots, and the **latest** value, with how many times the value **changed**;
* a **status** that keeps four things apart: *present, above zero*, *present, zero* (PDC reported 0), *missing* (no item was published) and *conflicting* (two different values for the same snapshot time; neither is picked and the peak is left empty).

Exposure is **never added** across snapshots, events, categories or countries: repeated PDC alerts can cover the same people. A *peak* is one snapshot's value, not a total.

In [ ]:
#@title Functions: family_row, build_families
# `family_row` describes one family from its snapshots; `build_families` does that for every family key.
def family_row(key, group, impact_counts):
    """One event family from its snapshots."""
    sources, corr_ids = {value for value in group["src_event_id"] if value}, {value for value in group["corr_id"] if value}
    source_id = next(iter(sources)) if len(sources) == 1 else None
    if source_id:
        exposure_items = impact_counts["source"].get(source_id, 0)
    else:
        exposure_items = impact_counts["corr"].get(next(iter(corr_ids)), 0) if len(corr_ids) == 1 else 0
    starts = [v for v in group["start_datetime"].where(group["start_datetime"].notna(), group["event_datetime"]) if v]
    ends = [v for v in group["end_datetime"].where(group["end_datetime"].notna(), group["event_datetime"]) if v]
    codes = tuple(sorted({code for codes in group["hazard_codes"] for code in codes}))
    return {
        "family_key": key, "source_event_id": source_id, "event_title": next((title for title in group["event_title"] if title), None),
        "start_datetime": min(starts) if starts else None, "end_datetime": max(ends) if ends else None,
        "country_codes": tuple(sorted({code for codes in group["country_codes"] for code in codes})), "hazard_codes": codes,
        "hazard_labels": hazard_groups(codes), "snapshot_count": group["event_item_id"].nunique(), "impact_record_count": exposure_items,
        "geometry_types": tuple(sorted({kind for kind in group["geometry_type"] if kind})), "provider": "api",
        "correlation_status": "exact_source_id" if source_id else "legacy_corr_id_fallback" if corr_ids else "unlinked",
    }


def build_families(events, impacts):
    """One row per event family: all snapshots with the same family key."""
    impact_counts = {"source": impacts.dropna(subset=["src_event_id"]).groupby("src_event_id")["impact_item_id"].nunique().to_dict(),
                     "corr": impacts.dropna(subset=["corr_id"]).groupby("corr_id")["impact_item_id"].nunique().to_dict()}
    return pd.DataFrame([family_row(key, group, impact_counts) for key, group in events.sort_values("event_item_id").groupby("family_key", sort=True)])

In [ ]:
#@title Functions: alert_rank, measure_stats
SUMMARY_MEASURES = ("people", "households", "schools", "hospitals", "capital")
ALERT_LEVELS = ("INFORMATION", "ADVISORY", "WATCH", "WARNING")
SUMMARY_COLUMNS = [
    "family_key", "source_event_id", "title", "pdc_hazard_type", "hazard_group", "hazard_groups", "hazard_codes", "caveat", "countries",
    "n_countries", "multi_country", "country", "event_date", "year", "month", "start", "end", "duration_hours", "snapshot_count",
    "first_snapshot_time", "last_snapshot_time", "alert_level_max", "alert_level_latest", "longitude", "latitude", "valid_point",
    "footprint_url", "report_url", "description_currency_quote",
    *(f"{measure}_{kind}" for measure in SUMMARY_MEASURES for kind in ("peak", "peak_time", "latest", "latest_time", "changes", "status")),
]


def alert_rank(label):
    """Position of a PDC alert level (information < advisory < watch < warning); 0 when absent or unknown."""
    text = str(label).upper() if isinstance(label, str) else ""
    return ALERT_LEVELS.index(text) + 1 if text in ALERT_LEVELS else 0


def measure_stats(impacts):
    """For every (event family, category): rows, conflict flag, peak, latest and number of changes."""
    keys = ["family_key", "impact_category"]
    data = impacts[impacts["impact_category"].isin(set(MEASURE_CATEGORY.values()))].copy()
    if data.empty:
        return pd.DataFrame()
    data["value"] = pd.to_numeric(data["numeric_value"], errors="coerce")
    data["time"] = data["snapshot_time"].fillna(data["item_datetime"])  # the exposure time, else the event time
    rows = data.groupby(keys).size().rename("rows")
    per_time = (data.dropna(subset=["value"]).groupby([*keys, "time"], dropna=False)["value"].agg(["nunique", "max"])
                .rename(columns={"max": "value"}).reset_index().sort_values([*keys, "time"], kind="stable"))
    conflict = per_time.groupby(keys)["nunique"].max().gt(1).rename("conflict")  # two different values at one snapshot time
    peak = per_time.loc[per_time.groupby(keys)["value"].idxmax()].set_index(keys)[["value", "time"]].set_axis(["peak", "peak_time"], axis=1)
    latest = per_time.groupby(keys).tail(1).set_index(keys)[["value", "time"]].set_axis(["latest", "latest_time"], axis=1)
    changes = per_time.groupby(keys)["value"].agg(lambda values: int(values.diff().fillna(0).ne(0).sum())).rename("changes")
    return pd.concat([rows, conflict, peak, latest, changes], axis=1)

In [ ]:
#@title Functions: start_summary, add_dates
# Assembling the summary takes five small steps, each its own function: the basics, the dates, the PDC alert levels, the location, and the
# measures.
def start_summary(country, families):
    """Title, hazard group, caveat and countries of each family."""
    summary = pd.DataFrame({
        "family_key": families["family_key"], "source_event_id": families["source_event_id"], "title": families["event_title"],
        "pdc_hazard_type": families["event_title"].map(pdc_hazard_type), "hazard_group": families["hazard_codes"].map(hazard_group),
        "hazard_groups": families["hazard_labels"], "hazard_codes": families["hazard_codes"],
        "caveat": [code_title_caveat(title, codes) for title, codes in zip(families["event_title"], families["hazard_codes"])],
        "countries": families["country_codes"],
    })
    summary["n_countries"] = summary["countries"].map(len)
    summary["multi_country"] = summary["n_countries"].gt(1)
    summary["country"] = country
    return summary


def add_dates(summary, families, events):
    """Event start (the earliest event datetime of the snapshots), the event span and the first and last exposure snapshot times."""
    event_dates = pd.to_datetime(events.groupby("family_key")["event_datetime"].min(), errors="coerce", utc=True, format="mixed")
    summary["event_date"] = summary["family_key"].map(event_dates)
    summary["year"], summary["month"] = summary["event_date"].dt.year.astype("Int64"), summary["event_date"].dt.month.astype("Int64")
    summary["start"] = pd.to_datetime(families["start_datetime"], errors="coerce", utc=True, format="mixed")
    summary["end"] = pd.to_datetime(families["end_datetime"], errors="coerce", utc=True, format="mixed")
    summary["duration_hours"] = (summary["end"] - summary["start"]).dt.total_seconds() / 3600
    summary["snapshot_count"] = families["snapshot_count"].astype(int)
    snapshot_times = pd.to_datetime(events["snapshot_time"], errors="coerce", utc=True, format="mixed")
    summary["first_snapshot_time"] = summary["family_key"].map(snapshot_times.groupby(events["family_key"]).min())
    summary["last_snapshot_time"] = summary["family_key"].map(snapshot_times.groupby(events["family_key"]).max())
    return summary

In [ ]:
#@title Functions: add_alerts, add_location
def add_alerts(summary, hazards):
    """The highest and the latest PDC alert level of each family (the hazard items' severity label)."""
    if hazards.empty:
        summary["alert_level_max"] = summary["alert_level_latest"] = None
        return summary
    ranked = hazards.assign(rank=hazards["severity_label"].map(alert_rank), order=hazards["snapshot_time"].fillna(hazards["item_datetime"]).fillna(""))
    ranked = ranked[ranked["rank"] > 0]
    summary["alert_level_max"] = summary["family_key"].map(ranked.groupby("family_key")["rank"].max().map(lambda rank: ALERT_LEVELS[rank - 1]))
    summary["alert_level_latest"] = summary["family_key"].map(
        ranked.sort_values(["family_key", "order"], kind="stable").groupby("family_key")["severity_label"].last().str.upper())
    return summary


def add_location(summary, events):
    """Location, links and wording come from the family's latest snapshot."""
    events = events.assign(order=events["snapshot_time"].fillna(events["event_datetime"]).fillna(""))
    latest = events.sort_values(["family_key", "order", "event_item_id"], kind="stable").groupby("family_key").tail(1).set_index("family_key")
    for column, source in (("longitude", "longitude"), ("latitude", "latitude"), ("valid_point", "valid_geometry"), ("footprint_url", "footprint_url"), ("report_url", "report_url")):
        summary[column] = summary["family_key"].map(latest[source])
    summary["valid_point"] = summary["valid_point"].fillna(False).astype(bool)
    quotes = events.dropna(subset=["description_currency_quote"]).sort_values("order").groupby("family_key")["description_currency_quote"].last()
    summary["description_currency_quote"] = summary["family_key"].map(quotes)
    return summary

In [ ]:
#@title Functions: measure_columns, build_summary
def measure_columns(summary, stats, measure):
    """Peak, latest, changes and status of one measure for every family. A conflict leaves the peak empty: neither value is picked."""
    category = MEASURE_CATEGORY[measure]
    found = stats.xs(category, level="impact_category") if len(stats) and category in stats.index.get_level_values("impact_category") else pd.DataFrame()
    empty = pd.DataFrame(index=summary["family_key"], columns=["rows", "conflict", "peak", "peak_time", "latest", "latest_time", "changes"])
    mapped = found.reindex(summary["family_key"]) if len(found) else empty
    conflict = mapped["conflict"].eq(True).to_numpy()
    peak = pd.to_numeric(mapped["peak"], errors="coerce").to_numpy(dtype=float)
    summary[f"{measure}_peak"] = np.where(conflict, np.nan, peak)
    summary[f"{measure}_peak_time"] = pd.to_datetime(mapped["peak_time"].to_numpy(), errors="coerce", utc=True, format="mixed")
    summary[f"{measure}_latest"] = np.where(conflict, np.nan, pd.to_numeric(mapped["latest"], errors="coerce").to_numpy(dtype=float))
    summary[f"{measure}_latest_time"] = pd.to_datetime(mapped["latest_time"].to_numpy(), errors="coerce", utc=True, format="mixed")
    summary[f"{measure}_changes"] = pd.to_numeric(mapped["changes"], errors="coerce").fillna(0).astype(int).to_numpy()
    rows = pd.to_numeric(mapped["rows"], errors="coerce").fillna(0).to_numpy()
    summary[f"{measure}_status"] = np.select([np.full(len(summary), category not in CATEGORIES), rows == 0, conflict, np.isnan(peak), peak == 0],
                                             ["not_retrieved", "missing", "conflicting", "unavailable", "present_zero"], default="present_positive")


def build_summary(country, families, events, hazards, impacts):
    """One row per event family with peak and latest exposure per measure."""
    summary = add_location(add_alerts(add_dates(start_summary(country, families), families, events), hazards), events)
    stats = measure_stats(impacts)
    for measure in SUMMARY_MEASURES:
        measure_columns(summary, stats, measure)
    return summary[SUMMARY_COLUMNS].reset_index(drop=True)

In [ ]:
#@title Functions: process_country
# Run it for every country
def process_country(country):
    """Item tables, links, families and summary for one country."""
    source = RAW[country]
    events = pd.DataFrame([tidy_event(v["item"], v, country) for v in source["pdc-events"].values()], columns=EVENT_COLUMNS)
    hazards = pd.DataFrame([tidy_hazard(v["item"], v, country) for v in source["pdc-hazards"].values()], columns=HAZARD_COLUMNS)
    impacts = pd.DataFrame([row for v in source["pdc-impacts"].values() for row in tidy_impacts(v["item"], v, country)], columns=IMPACT_COLUMNS)
    edges = link_evidence(country)
    events = events.sort_values("event_item_id").reset_index(drop=True)
    hazards = attach_links(hazards.sort_values("hazard_item_id").reset_index(drop=True), "hazard_item_id", edges, "pdc-hazards")
    impacts = attach_links(impacts.sort_values(["impact_item_id", "detail_position"]).reset_index(drop=True), "impact_item_id", edges, "pdc-impacts")
    events["correlation_status"] = np.where(events["family_key"].str.startswith("item:"), "unlinked", "grouped")
    tables = {"events": events, "hazards": hazards, "impacts": impacts, "edges": edges, "families": pd.DataFrame(), "summary": pd.DataFrame(columns=SUMMARY_COLUMNS)}
    if len(events):
        tables["families"] = build_families(events, impacts)
        tables["summary"] = build_summary(country, tables["families"], events, hazards, impacts)
    return tables


TABLES = {country: process_country(country) for country in COUNTRIES}
SUMMARIES = {country: tables["summary"] for country, tables in TABLES.items()}

In [ ]:
counts = pd.DataFrame({country_name(country): {
    "event families": len(t["families"]), "event snapshots": len(t["events"]), "hazard items": len(t["hazards"]),
    "exposure values": len(t["impacts"]), "links checked": len(t["edges"]),
} for country, t in TABLES.items()}).T
display(counts)
if len(INCOMPLETE):
    display(HTML(f"<p style='color:#B8312F'><b>The retrieval was incomplete</b> ({len(INCOMPLETE)} window(s) missing), so these tables hold only what was read. See step 5.</p>"))

first = COUNTRIES[0]
print(f"Preview for {country_name(first)} (first rows only; the full tables are in TABLES['{first}']):")
for name, meaning in (("events", "event snapshot"), ("hazards", "hazard item"), ("impacts", "exposure value")):
    print(f"\n{name}: one row per {meaning}")
    display(TABLES[first][name].drop(columns=["geometry"]).head(3))

print("summary: one row per event family, with peak and latest exposure per measure")
display(SUMMARIES[first][["title", "hazard_group", "event_date", "snapshot_count", "people_peak", "people_latest", "people_status", "alert_level_max"]].head(5))

## 7. Choose the view

The sections below show **one country** and **one measure** at a time. Choose them here. After you change a picker, run the sections below (steps 8 to 14) again:
this only redraws charts from the tables already in memory, **no API call is made**.
Tsunami bulletins are alerts for a whole warning region (their point is the warning-centre location), so they are hidden by default.

In [ ]:
#@title Functions: is_bulletin, view
MEASURE_SHORT = {"people": "People", "households": "Households", "schools": "Schools", "hospitals": "Hospitals", "capital": "Capital"}
MEASURE_LABELS = {"people": "People exposed", "households": "Households exposed", "schools": "Schools exposed",
                  "hospitals": "Hospitals exposed", "capital": "Capital exposure (unit not stated)"}
MEASURE_NOUNS = {"people": "people", "households": "households", "schools": "schools", "hospitals": "hospitals", "capital": "capital value"}

country_picker = widgets.Dropdown(options=[(country_name(code), code) for code in COUNTRIES], description="Country", style={"description_width": "70px"})
measure_picker = widgets.Dropdown(options=[(MEASURE_SHORT[name], name) for name in MEASURE_NAMES], value="people", description="Measure", style={"description_width": "70px"})
bulletin_toggle = widgets.Checkbox(value=False, description="Show tsunami bulletins", indent=False)
display(widgets.HBox([country_picker, measure_picker, bulletin_toggle], layout=widgets.Layout(gap="24px")))


def is_bulletin(summary):
    return summary["caveat"].fillna("").str.startswith("Tsunami bulletin")


def view():
    """The country, measure and events chosen above. Changing a picker never calls the API."""
    country, measure = country_picker.value, measure_picker.value
    summary = SUMMARIES[country]
    if not bulletin_toggle.value and len(summary):
        summary = summary[~is_bulletin(summary)]
    return country, measure, summary.reset_index(drop=True)


period_label = (str(YEARS[0]) if len(YEARS) == 1 else f"{YEARS[0]}–{YEARS[-1]}") + " · " + ("Jan–Dec" if len(REQUESTED_MONTHS) == 12 else ", ".join(MONTH_NAMES[m - 1] for m in REQUESTED_MONTHS))
print(f"Showing {country_name(country_picker.value)}, {MEASURE_SHORT[measure_picker.value].lower()}, {period_label}.")

In [ ]:
#@title Functions: say, note, cards
# Small helpers for the charts
def say(title, subtitle=""):
    """A question above a chart, with the grain and period in the subtitle."""
    sub = f"<div style='opacity:.75;font-size:.85rem'>{escape(subtitle)}</div>" if subtitle else ""
    display(HTML(f"<div style='margin:16px 0 2px'><div style='font-size:1.05rem;font-weight:650'>{escape(title)}</div>{sub}</div>"))


def note(text):
    display(HTML(f"<p style='opacity:.75;font-size:.82rem;line-height:1.4;margin:2px 0 8px'>{escape(text)}</p>"))


def cards(items):
    """Headline numbers: (label, value, small print) per card."""
    html = "".join(
        "<div style='border:1px solid rgba(128,128,128,.35);border-radius:10px;padding:10px 14px;background:rgba(128,128,128,.07);min-width:0'>"
        f"<div style='font-size:.72rem;font-weight:600;letter-spacing:.04em;text-transform:uppercase;opacity:.7'>{escape(label)}</div>"
        f"<div style='font-size:1.5rem;font-weight:700;line-height:1.3'>{escape(value)}</div>"
        f"<div style='font-size:.78rem;opacity:.75'>{escape(small)}</div></div>" for label, value, small in items)
    display(HTML(f"<div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(190px,1fr));gap:10px;margin:6px 0'>{html}</div>"))

In [ ]:
#@title Functions: show, interactive, short, …
def show(fig):
    fig.show(config=PLOTLY_CONFIG)


def interactive(controls, draw, *fixed):
    """Show controls above a chart area; changing a control redraws only that area, with `draw(*fixed, *control values)` (no API call)."""
    area = widgets.Output()

    def redraw(_change=None):
        with area:
            area.clear_output(wait=True)
            draw(*fixed, *[control.value for control in controls])

    for control in controls:
        control.observe(redraw, names="value")
    display(widgets.VBox([widgets.HBox(controls, layout=widgets.Layout(gap="16px")), area]))
    redraw()


def short(title, limit=46):
    text = str(title or "Untitled")
    return text if len(text) <= limit else text[: limit - 1].rstrip() + "…"


def date_text(value):
    return value.strftime("%d %b %Y") if isinstance(value, pd.Timestamp) and not pd.isna(value) else "date unknown"

In [ ]:
#@title Functions: positive, groups_present, hover_rows, …
def positive(summary, measure):
    """The events whose peak value of the measure is above zero (the only ones a log scale can draw)."""
    column = f"{measure}_peak"
    return summary[pd.to_numeric(summary[column], errors="coerce").gt(0)] if len(summary) else summary


def groups_present(data):
    """The hazard groups that occur in the data, in the fixed colour order."""
    present = set(data["hazard_group"].dropna()) if len(data) else set()
    return [group for group in HAZARD_GROUP_ORDER if group in present]


EVENT_HOVER = ("<b>%{customdata[1]}</b><br>%{customdata[6]} · %{customdata[2]}<br>"
               "Peak: %{customdata[3]} · Latest: %{customdata[4]}<br>Highest PDC alert: %{customdata[5]}<extra></extra>")


def hover_rows(data, measure):
    """customdata for the hover text: family key, title, date, peak, latest, alert level, hazard type."""
    return np.column_stack([data["family_key"].astype(str), data["title"].map(short), data["event_date"].map(date_text), data[f"{measure}_peak"].map(compact),
                            data[f"{measure}_latest"].map(compact), data["alert_level_max"].fillna("not stated"), data["pdc_hazard_type"].fillna(data["hazard_group"])])


def incomplete_banner():
    """A warning above the charts when part of the retrieval failed."""
    if len(INCOMPLETE):
        display(HTML("<div style='border:1px solid #EBCF97;background:rgba(235,207,151,.25);border-radius:10px;padding:8px 14px;margin:6px 0'>"
                     f"<b>Incomplete result.</b> {len(INCOMPLETE)} window(s) could not be read (see step 5), so the figures below show only what was retrieved.</div>"))

## 8. Overview

**How many events, and how big?** Each event family is counted **once**, in the month it started. Its size is the **peak** value of the chosen measure among its snapshots.
Exposure is not added across events, because repeated PDC alerts can cover the same people.

In [ ]:
country, measure, summary = view()
incomplete_banner()
if summary.empty:
    note("No events in this view. Change the country or the filters.")
else:
    values = pd.to_numeric(summary[f"{measure}_peak"], errors="coerce")
    warnings = int((summary["alert_level_max"] == "WARNING").sum())
    largest = summary.loc[values.idxmax()] if values.notna().any() else None
    cards([
        ("Events", f"{len(summary):,}", f"{country_name(country)} · {period_label}"),
        ("PDC updates", f"{int(summary['snapshot_count'].sum()):,}", "event snapshots; one event is updated many times"),
        ("Events listing other countries", f"{int(summary['multi_country'].sum()):,}", "regional alerts also counted in those countries"),
        ("Reached PDC 'Warning'", f"{warnings:,}", f"{warnings / len(summary):.0%} of events"),
        (f"Largest event · {MEASURE_NOUNS[measure]}", compact(largest[f"{measure}_peak"]) if largest is not None else "–",
         f"{short(largest['title'], 40)} · {date_text(largest['event_date'])}" if largest is not None else "no values"),
    ])

In [ ]:
#@title Functions: size_labels, size_class
SIZE_EDGES = {"people": (1e4, 1e5, 1e6), "households": (1e3, 1e4, 1e5), "schools": (10, 100, 1000), "hospitals": (5, 25, 100), "capital": (1e6, 1e8, 1e9)}
SIZE_COLORS = ("#F8B878", "#E57B4A", "#B93E3B", "#561C3A")  # four classes, smallest to largest
NO_VALUE_LABEL = "Zero or no value"


def size_labels(measure):
    low, mid, high = SIZE_EDGES[measure]
    return (f"Under {compact(low)}", f"{compact(low)}–{compact(mid)}", f"{compact(mid)}–{compact(high)}", f"{compact(high)} or more")


def size_class(values, measure):
    """A readable size class for each event's peak value."""
    labels, (low, mid, high) = size_labels(measure), SIZE_EDGES[measure]
    numbers = pd.to_numeric(values, errors="coerce")
    return pd.Series(np.select([numbers.isna() | numbers.le(0), numbers.lt(low), numbers.lt(mid), numbers.lt(high)],
                               [NO_VALUE_LABEL, labels[0], labels[1], labels[2]], default=labels[3]), index=values.index)

In [ ]:
#@title Functions: monthly_counts, monthly_hover
def monthly_counts(summary, measure, split):
    """Distinct events per month (by event start): one column per size class or hazard group. Months that were not requested are empty, not zero."""
    data = summary.dropna(subset=["event_date"]).copy()
    data["month"] = data["event_date"].dt.tz_convert(None).dt.to_period("M")
    data["group"] = data["hazard_group"] if split == "hazard" else size_class(data[f"{measure}_peak"], measure)
    months = pd.period_range(f"{YEARS[0]}-01", f"{YEARS[-1]}-12", freq="M")
    counts = data.groupby(["month", "group"])["family_key"].nunique().unstack(fill_value=0).reindex(months, fill_value=0).astype(float)
    counts.loc[[month.month not in REQUESTED_MONTHS for month in months]] = np.nan
    return counts


def monthly_hover(summary, measure):
    """For each month: how many events started and which was the largest (never a sum of exposure)."""
    data = summary.dropna(subset=["event_date"]).assign(month=lambda d: d["event_date"].dt.tz_convert(None).dt.to_period("M"),
                                                        value=lambda d: pd.to_numeric(d[f"{measure}_peak"], errors="coerce"))
    largest = data.dropna(subset=["value"]).sort_values("value").groupby("month").tail(1).set_index("month")
    return {month: f"{total} events" + (f" · largest: {short(largest.loc[month, 'title'], 38)} ({compact(largest.loc[month, 'value'])} {MEASURE_NOUNS[measure]})"
                                        if month in largest.index else "") for month, total in data.groupby("month")["family_key"].nunique().items()}

In [ ]:
#@title Functions: fig_events_per_month
def fig_events_per_month(counts, hover, split, measure):
    """A stacked area of distinct events per month, with a hover line that names each month's largest event."""
    sizes = size_labels(measure)
    order = [g for g in HAZARD_GROUP_ORDER if g in counts.columns] if split == "hazard" else [c for c in [*sizes[::-1], NO_VALUE_LABEL] if c in counts.columns]
    colors = {g: hazard_color(g) for g in order} if split == "hazard" else {**dict(zip(sizes, SIZE_COLORS, strict=True)), NO_VALUE_LABEL: "#D5D9E0"}
    x, fig = counts.index.to_timestamp(), go.Figure()
    for name in order:
        label = name if split == "hazard" or name == NO_VALUE_LABEL else f"{name} {MEASURE_NOUNS[measure]}"
        fig.add_scatter(x=x, y=counts[name], name=label, mode="lines", stackgroup="events", line=dict(width=0.8, color=colors[name]),
                        fillcolor=with_alpha(colors[name], 0.88), hovertemplate="%{y:.0f}<extra>" + label + "</extra>")
    totals = counts.sum(axis=1, min_count=1)  # a month that was not requested stays empty
    fig.add_scatter(x=x, y=totals, mode="lines", line=dict(width=0, color="rgba(0,0,0,0)"), showlegend=False, hovertemplate="%{customdata}<extra>Month</extra>",
                    customdata=["month not requested" if np.isnan(totals[m]) else hover.get(m, "0 events") for m in counts.index])
    fig.update_layout(hovermode="x unified", legend=dict(traceorder="normal", font=dict(size=12, color=INK), itemsizing="constant"))
    fig.update_yaxes(title="Events per month", rangemode="tozero", fixedrange=True)
    fig.update_xaxes(type="date", showgrid=False, tickformat="%b %Y" if len(YEARS) > 1 else "%b", **({} if len(YEARS) > 1 else {"dtick": "M1"}))
    for year in YEARS[1:]:
        fig.add_vline(x=pd.Timestamp(f"{year}-01-01").timestamp() * 1000 - 15 * 86400000, line=dict(color="#C9CED6", width=1, dash="dot"))
    return apply_plotly_theme(fig, height=400 if len(YEARS) > 1 else 360)

In [ ]:
#@title Functions: draw_monthly
def draw_monthly(summary, measure, split):
    """Prepare the monthly counts and draw the chart for the chosen split."""
    if summary.dropna(subset=["event_date"]).empty:
        return show(empty_figure("No dated events in this result."))
    show(fig_events_per_month(monthly_counts(summary, measure, split), monthly_hover(summary, measure), split, measure))


country, measure, summary = view()
say("Events per month", f"Distinct events by month of event start, {country_name(country)}. Split the stack by size or by hazard:")
split_picker = widgets.ToggleButtons(options=[("By size", "size"), ("By hazard", "hazard")], value="size")
interactive([split_picker], draw_monthly, summary, measure)
note("Each event is counted once, in the month it started. PDC coverage in Montandon differs strongly between years (see step 14): differences can reflect coverage, not only hazard activity.")

In [ ]:
#@title Functions: season_grid, fig_seasonality
# When in the year do events start?
def season_grid(summary):
    """Distinct events for every year × month; None where a month was not requested."""
    counts = summary.dropna(subset=["year", "month"]).groupby(["year", "month"])["family_key"].nunique()
    return [[None if month not in REQUESTED_MONTHS else int(counts.get((year, month), 0)) for month in range(1, 13)] for year in YEARS]


def fig_seasonality(grid):
    """A year-by-month calendar. Darker = more events; months that were not requested are blank."""
    text = [["" if value in (None, 0) else str(value) for value in row] for row in grid]
    hover = [[f"{MONTH_NAMES[m]} {year}: " + ("not requested" if value is None else f"{value} event{'s' if value != 1 else ''}")
              for m, value in enumerate(row)] for year, row in zip(YEARS, grid, strict=True)]
    peak = max((value for row in grid for value in row if value is not None), default=0) or 1
    fig = go.Figure(go.Heatmap(z=grid, x=list(MONTH_NAMES), y=[str(year) for year in YEARS], zmin=0, zmax=peak, colorscale=sequential_scale(), xgap=3, ygap=3,
                               showscale=False, customdata=hover, hovertemplate="%{customdata}<extra></extra>", text=text, texttemplate="%{text}", textfont=dict(size=12)))
    fig.update_yaxes(autorange="reversed", type="category", showgrid=False, fixedrange=True)
    fig.update_xaxes(side="top", showgrid=False, fixedrange=True)
    return apply_plotly_theme(fig, height=64 + 40 * len(YEARS))


country, measure, summary = view()
say("When in the year do events start?", "Distinct events per month. Darker = more events.")
show(fig_seasonality(season_grid(summary)) if len(summary) else empty_figure("No events in the selected period."))

In [ ]:
#@title Functions: wrap_label, box_labels
# Which hazards expose the most per event?
MIN_BOX_EVENTS = 5  # fewer events than this: a range line and the median instead of a box


def wrap_label(text, width=12):
    """Break a label into <br>-separated lines of at most `width` characters without splitting words."""
    lines = []
    for word in str(text).split():
        if lines and len(lines[-1]) + 1 + len(word) <= width:
            lines[-1] = f"{lines[-1]} {word}"
        else:
            lines.append(word)
    return "<br>".join(lines)


def box_labels(data, summary, order):
    """Each hazard's axis label: its name and how many of its events have a value."""
    totals = summary.groupby("hazard_group")["family_key"].nunique()
    labels = {}
    for group in order:
        count, total = int((data["hazard_group"] == group).sum()), int(totals.get(group, 0))
        detail = f"{count} events" if count == total else f"{count} of {total}<br>events"
        labels[group] = f"{wrap_label(group)}<br><span style='font-size:10.5px;color:{INK_SECONDARY}'>{detail}</span>"
    return labels

In [ ]:
#@title Functions: add_hazard_box, fig_hazard_profile
def add_hazard_box(fig, label, group, values, measure):
    """The box (or, for few events, range and median) of one hazard group."""
    color, median = hazard_color(group), values.median()
    q1, q3 = values.quantile([0.25, 0.75])
    if len(values) >= MIN_BOX_EVENTS:
        fig.add_trace(go.Box(x=[label] * len(values), y=values, name=group, width=0.45, boxpoints=False, fillcolor=with_alpha(color, 0.28),
                             line=dict(color=color, width=1.8), whiskerwidth=0.4, hoveron="boxes", hoverinfo="skip", showlegend=False))
    else:
        fig.add_trace(go.Scatter(x=[label] * 2, y=[values.min(), values.max()], mode="lines", line=dict(color=color, width=2.5), hoverinfo="skip", showlegend=False))
    fig.add_trace(go.Scatter(x=[label], y=[median], mode="markers", marker=dict(symbol="line-ew", size=34, line=dict(width=3, color=INK)), showlegend=False,
                             hovertemplate=(f"<b>{group}</b><br>Median: {compact(10 ** median)} {MEASURE_NOUNS[measure]} per event<br>Middle half: {compact(10 ** q1)} – "
                                            f"{compact(10 ** q3)}<br>Range: {compact(10 ** values.min())} – {compact(10 ** values.max())}<extra></extra>")))


def fig_hazard_profile(summary, measure):
    """One box plot per hazard of the peak exposure per event (log scale), ordered by median."""
    data = positive(summary, measure)
    if data.empty:
        return empty_figure(f"No event had {MEASURE_NOUNS[measure]} exposed.", height=420)
    data = data.assign(_log=np.log10(pd.to_numeric(data[f"{measure}_peak"]).astype(float)))  # quartiles on log10 values; order statistics do not change
    order = list(data.groupby("hazard_group")["_log"].median().sort_values(ascending=False).index)
    labels, fig = box_labels(data, summary, order), go.Figure()
    for group in order:
        add_hazard_box(fig, labels[group], group, data.loc[data["hazard_group"] == group, "_log"], measure)
    low, high = float(data["_log"].min()), float(data["_log"].max())
    ticks = list(range(math.floor(low), math.ceil(high) + 1))
    fig.update_yaxes(tickvals=ticks, ticktext=[compact(10**power) for power in ticks], range=[low - 0.25, high + 0.25], title=MEASURE_LABELS[measure] + " per event (peak, log scale)")
    fig.update_xaxes(type="category", categoryorder="array", categoryarray=[labels[g] for g in order], showgrid=False, title=None, tickfont=dict(size=10.5, color=INK), tickangle=0, automargin=True)
    fig.update_layout(boxgap=0.3)
    return apply_plotly_theme(fig, height=420)


country, measure, summary = view()
say(f"Which hazards expose the most {MEASURE_NOUNS[measure]} per event?", "Peak exposure per event, one box per hazard group, ordered by median.")
show(fig_hazard_profile(summary, measure))
note("Only events with a value above zero are drawn (log scale); each label says how many of the hazard's events that is. Hover a box for its exact statistics.")

## 9. Map

**Where are the events?** One dot per **event family**, at the PDC event point. Colour is the hazard group; the diameter grows by a fixed step per tenfold increase in peak exposure
(log scale), so a 1M-person event is clearly bigger than a 10k one. Smaller dots are drawn on top, and nothing is jittered.
Alerts that list several countries are drawn faded, because their PDC point is only a reference location. The **Clusters** layer groups nearby events into numbered circles
(the number is how many *distinct events* they hold, never a sum of exposure).

> **PDC point represents the event/hazard location; it is not automatically an impact footprint.** The footprint of one event is shown in step 12.

Two small functions choose the part of the world to show: `fit_view` finds the centre and zoom for a set of points, and `map_frame` decides which points to frame.

In [ ]:
#@title Functions: fit_view, map_frame
CLUSTER_MAX_ZOOM = 6  # nearby events merge into count circles up to this zoom level


def fit_view(lons, lats, trim):
    """Map centre, zoom and which points fall inside: the central 90% of the points (all of them when there are few)."""
    if trim and len(lons) >= 10:
        lon_low, lon_high = np.quantile(lons, [0.05, 0.95])
        lat_low, lat_high = np.quantile(lats, [0.05, 0.95])
    else:
        lon_low, lon_high, lat_low, lat_high = lons.min(), lons.max(), lats.min(), lats.max()
    pad_lon, pad_lat = max((lon_high - lon_low) * 0.12, 0.6), max((lat_high - lat_low) * 0.12, 0.6)
    lon_low, lon_high, lat_low, lat_high = lon_low - pad_lon, lon_high + pad_lon, lat_low - pad_lat, lat_high + pad_lat
    inside = (lons >= lon_low) & (lons <= lon_high) & (lats >= lat_low) & (lats <= lat_high)
    span = max(lon_high - lon_low, (lat_high - lat_low) * 1.6, 0.05)
    return {"lon": float((lon_low + lon_high) / 2), "lat": float((lat_low + lat_high) / 2)}, float(np.clip(math.log2(360 / span) + 0.6, 1, 11)), inside


def map_frame(points, fit_all):
    """Centre, zoom and the points inside the view. By default the view frames single-country events, so a few distant regional alerts do not shrink the map."""
    lons, lats = points["longitude"].to_numpy(float), points["latitude"].to_numpy(float)
    local = ~points["multi_country"].to_numpy(bool)
    if fit_all or local.sum() < 3:
        return fit_view(lons, lats, trim=not fit_all)
    center, zoom, _ = fit_view(lons[local], lats[local], trim=False)
    span_lon = 360 / 2 ** (zoom - 0.6)
    inside = (np.abs(lons - center["lon"]) <= span_lon / 2) & (np.abs(lats - center["lat"]) <= span_lon / 1.6 / 2)
    return center, zoom, inside

In [ ]:
#@title Functions: map_points, map_markers, fig_event_map
def map_points(summary):
    """Events with a valid PDC point (one dot per event family)."""
    return summary[summary["valid_point"]] if len(summary) else summary


def map_markers(points, measure, uniform_size=None):
    """Marker data: biggest first, so small events are drawn on top. Size = log-scaled peak value; colour = hazard group, faded for multi-country alerts."""
    values = pd.to_numeric(points[f"{measure}_peak"], errors="coerce")
    sizes = values.map(exposure_size) if uniform_size is None else pd.Series(float(uniform_size), index=values.index)
    data = points.assign(_size=sizes, _value=values.fillna(-1)).sort_values("_value", ascending=False, kind="stable")
    data["_color"] = [with_alpha(hazard_color(group), 0.38 if regional else 0.9) for group, regional in zip(data["hazard_group"], data["multi_country"], strict=True)]
    return data


def fig_event_map(summary, measure, layer="points", fit_all=False, uniform_size=None, zoom_offset=0.0, height=640):
    """One marker per event family; `layer` is "points" or "clusters"; `uniform_size` draws every dot at one size."""
    points = map_points(summary)
    if points.empty:
        return empty_figure("No events with a valid PDC point in this view.", height=260)
    center, zoom, _ = map_frame(points, fit_all)
    data, fig = map_markers(points, measure, uniform_size), go.Figure()
    if layer == "points":  # a white halo under each marker separates overlapping points without jitter
        fig.add_trace(go.Scattermap(lat=data["latitude"], lon=data["longitude"], mode="markers", marker=dict(size=data["_size"] + 2.5, color=SURFACE), hoverinfo="skip", showlegend=False))
    fig.add_trace(go.Scattermap(
        lat=data["latitude"], lon=data["longitude"], mode="markers", marker=dict(size=data["_size"], color=data["_color"]), customdata=hover_rows(data, measure),
        hovertemplate=EVENT_HOVER, showlegend=False, name="events",
        cluster=dict(enabled=True, maxzoom=CLUSTER_MAX_ZOOM, color=with_alpha(PRIMARY, 0.82), size=[18, 26, 36], step=[10, 100]) if layer == "clusters" else None))
    for group in groups_present(data):  # legend: hazard colours only
        fig.add_trace(go.Scattermap(lat=[None], lon=[None], mode="markers", marker=dict(size=10, color=hazard_color(group)), name=group, hoverinfo="skip"))
    fig.update_layout(map=dict(style="carto-positron", center=center, zoom=max(zoom + zoom_offset, 0.5)), margin=dict(l=6, r=6, t=6, b=6),
                      legend=dict(orientation="h", yanchor="top", y=-0.02, xanchor="left", x=0, bgcolor="rgba(0,0,0,0)", font=dict(size=11.5), itemclick=False, itemdoubleclick=False))
    return apply_plotly_theme(fig, height=height)

In [ ]:
#@title Functions: size_key, map_caption
def size_key(summary, measure):
    """The marker-size key, drawn in HTML at the true dot diameters (a Plotly legend caps symbols at 16 px)."""
    values = pd.to_numeric(map_points(summary)[f"{measure}_peak"], errors="coerce")
    values = values[values > 0]
    if values.empty:
        return
    references = [value for value in SIZE_LEGEND_VALUES if values.min() / 10 < value <= values.max() * 3] or [SIZE_LEGEND_VALUES[0]]
    dots = "".join(f"<div style='display:flex;flex-direction:column;align-items:center;gap:3px;font-size:.76rem;opacity:.8'>"
                   f"<div style='width:{exposure_size(v):.1f}px;height:{exposure_size(v):.1f}px;border-radius:50%;background:rgba(128,122,115,.18);"
                   f"border:1.5px solid rgba(128,128,128,.9);box-sizing:border-box'></div>{compact(v)}</div>" for v in references)
    display(HTML(f"<div style='display:flex;align-items:flex-end;gap:18px;flex-wrap:wrap;margin:6px 0 4px 4px'>"
                 f"<div style='font-size:.8rem;font-weight:600;align-self:center'>{escape(MEASURE_LABELS[measure])} (peak)</div>{dots}</div>"))


def map_caption(summary, fit_all, layer):
    """What is on the map, what is not, and what a point means."""
    points = map_points(summary)
    parts = [f"{len(points)} events on the map."]
    if len(summary) and (~summary["valid_point"]).sum():
        parts.append(f"{int((~summary['valid_point']).sum())} without a valid PDC point are not drawn (they are in the events table).")
    if len(points) and (~map_frame(points, fit_all)[2]).sum():
        parts.append(f"{int((~map_frame(points, fit_all)[2]).sum())} far-away events are outside the view (tick 'Fit all events').")
    if len(points) and points["multi_country"].sum():
        parts.append(f"{int(points['multi_country'].sum())} multi-country alerts are drawn faded.")
    return " ".join(parts) + " " + MAP_NOTE + (" " + CLUSTER_NOTE if layer == "clusters" else "")

In [ ]:
#@title Functions: draw_map
def draw_map(summary, measure, layer, fit_all):
    """Draw the map, its size key and its caption."""
    show(fig_event_map(summary, measure, layer=layer, fit_all=fit_all))
    if layer == "points":
        size_key(summary, measure)
    note(map_caption(summary, fit_all, layer))


country, measure, summary = view()
say(f"Where did {country_name(country)}'s events happen?", f"One dot per event at its PDC event point; size = peak {MEASURE_NOUNS[measure]} exposed, colour = hazard.")
layer_picker = widgets.ToggleButtons(options=[("Events", "points"), ("Clusters", "clusters")], value="points")
fit_picker = widgets.Checkbox(value=False, description="Fit all events", indent=False)
interactive([layer_picker, fit_picker], draw_map, summary, measure)

## 10. Events

**Which events were largest, and when?** The bar chart ranks the 15 events with the largest peak of the chosen measure. The timeline puts every event on a time axis with one lane per hazard group, and dot size is the peak exposure.
The table lists the largest events with the PDC alert level and how many times PDC updated each one.

In [ ]:
#@title Functions: fig_top_events
def fig_top_events(summary, measure, n=15):
    """Horizontal bars of the largest events by peak value, coloured by hazard group."""
    data = positive(summary, measure)
    if data.empty:
        return empty_figure(f"No positive {MEASURE_NOUNS[measure]} values in this result.")
    column = f"{measure}_peak"
    top = data.nlargest(n, column).iloc[::-1]
    keys = list(top["family_key"].astype(str))  # the key is unique, so two events with the same title keep separate rows
    labels = [f"{short(title, 40)} · {date_text(date)}" for title, date in zip(top["title"], top["event_date"], strict=True)]
    fig = go.Figure()
    for group in groups_present(top):
        rows = top[top["hazard_group"] == group]
        fig.add_bar(x=rows[column], y=rows["family_key"].astype(str), orientation="h", name=group, marker=dict(color=hazard_color(group)), text=rows[column].map(compact),
                    textposition="outside", textfont=dict(color=INK_SECONDARY, size=11), cliponaxis=False, customdata=hover_rows(rows, measure), hovertemplate=EVENT_HOVER)
    fig.update_yaxes(type="category", categoryorder="array", categoryarray=keys, tickvals=keys, ticktext=labels, showgrid=False, tickfont=dict(size=11, color=INK))
    fig.update_xaxes(title=MEASURE_LABELS[measure] + " (peak)", rangemode="tozero", exponentformat="B", range=[0, float(top[column].max()) * 1.15])
    fig.update_layout(barmode="stack", bargap=0.3)
    return apply_plotly_theme(fig, height=80 + 30 * len(top))


country, measure, summary = view()
say("Largest events", f"Top 15 events by peak {MEASURE_NOUNS[measure]} exposed. Hover a bar for details.")
show(fig_top_events(summary, measure))

In [ ]:
#@title Functions: lane_jitter, fig_timeline
def lane_jitter(key):
    """A small, repeatable vertical offset, so events of one hazard do not sit on a single line."""
    return (int(hashlib.sha256(key.encode("utf-8")).hexdigest()[:6], 16) / 0xFFFFFF - 0.5) * 0.6


def fig_timeline(summary, measure):
    """Every event on a time axis: one lane per hazard group, dot size = peak exposure."""
    data = summary.dropna(subset=["event_date"])
    if data.empty:
        return empty_figure("No dated events in this result.")
    groups = groups_present(data)
    lanes = {group: index for index, group in enumerate(reversed(groups))}
    sizes = pd.to_numeric(data[f"{measure}_peak"], errors="coerce").map(exposure_size)
    fig = go.Figure()
    for group in groups:
        rows = data[data["hazard_group"] == group]
        fig.add_scatter(x=rows["event_date"], y=[lanes[group] + lane_jitter(key) for key in rows["family_key"]], mode="markers", name=group,
                        marker=dict(color=hazard_color(group), size=sizes[rows.index], opacity=0.8, line=dict(color=SURFACE, width=0.8)),
                        customdata=hover_rows(rows, measure), hovertemplate=EVENT_HOVER)
    fig.update_yaxes(tickvals=list(lanes.values()), ticktext=list(lanes.keys()), range=[-0.6, len(lanes) - 0.4], showgrid=False, zeroline=False)
    fig.update_xaxes(showgrid=True, title=None)
    fig.update_layout(showlegend=False)
    return apply_plotly_theme(fig, height=90 + 46 * len(groups))


country, measure, summary = view()
say("Every event over time", f"One row per hazard group; dot size = peak {MEASURE_NOUNS[measure]} exposed.")
show(fig_timeline(summary, measure))

In [ ]:
#@title Functions: events_table
def events_table(summary, measure, rows=100):
    """The largest events as a table (the export holds every event)."""
    ordered = summary.sort_values(f"{measure}_peak", ascending=False, na_position="last").head(rows)
    name = MEASURE_SHORT[measure].lower()
    return pd.DataFrame({
        "Event": ordered["title"].fillna("Untitled"), "Hazard": ordered["hazard_group"], "Start": ordered["event_date"].dt.tz_convert(None).dt.date,
        f"Peak {name}": ordered[f"{measure}_peak"], f"Latest {name}": ordered[f"{measure}_latest"],
        "Highest PDC alert": ordered["alert_level_max"].fillna("–").str.title(), "PDC updates": ordered["snapshot_count"],
        "Countries": ordered["countries"].map(lambda codes: ", ".join(country_name(code) for code in codes)), "Caveat": ordered["caveat"].fillna(""),
    }).reset_index(drop=True)


country, measure, summary = view()
say("All events", f"Sorted by peak {MEASURE_NOUNS[measure]}; at most 100 rows are shown here.")
if summary.empty:
    note("No events in this view.")
else:
    table = events_table(summary, measure)
    numbers = [column for column in table.columns if column.startswith(("Peak", "Latest"))]
    display(table.style.format({column: "{:,.0f}" for column in numbers}, na_rep="–").hide(axis="index"))
    note(f"Showing {len(table)} of {len(summary)} events. Missing values are shown as –; they are not zero.")

## 11. Exposure

**How much did events expose?** Every value is one event's **peak** for one measure: never a sum. Four views: the largest event of each month, how many events exposed *at least X*, how large events are,
and (for schools and hospitals) the events that exposed the most. Capital exposure carries **no unit in the structured PDC data**; its values are shown as published and no currency is assumed.

In [ ]:
#@title Functions: largest_per_month
MEASURE_COLORS = {"people": "#0B7A80", "households": "#6B4BA1", "schools": "#D0611E", "hospitals": "#B8312F", "capital": "#55657A"}


def largest_per_month(summary, measure):
    """For every month of the period: the peak of that month's largest event, a marker size and a hover text naming the event."""
    data = summary.dropna(subset=["event_date"]).assign(month=lambda d: d["event_date"].dt.tz_convert(None).dt.to_period("M"),
                                                        value=lambda d: pd.to_numeric(d[f"{measure}_peak"], errors="coerce"))
    largest = data.dropna(subset=["value"]).sort_values("value").groupby("month").tail(1).set_index("month")
    months, rows = pd.period_range(f"{YEARS[0]}-01", f"{YEARS[-1]}-12", freq="M"), []
    for month in months:
        if month.month not in REQUESTED_MONTHS:
            rows.append((None, 0, "month not requested"))
        elif month in largest.index:
            event = largest.loc[month]
            rows.append((float(event["value"]), 7, f"<b>{short(event['title'], 44)}</b><br>{event['hazard_group']} · {compact(event['value'])} {MEASURE_NOUNS[measure]}"))
        else:
            rows.append((0.0, 0, "no event with a value"))
    return months, rows

In [ ]:
#@title Functions: fig_measure_timeseries
def fig_measure_timeseries(months, rows, measure, height=280):
    """A line of the largest single event's peak in each month (never a sum across events)."""
    color = MEASURE_COLORS[measure]
    y, sizes, hover = (list(values) for values in zip(*rows, strict=True))
    fig = go.Figure(go.Scatter(x=months.to_timestamp(), y=y, mode="lines+markers", line=dict(color=color, width=2), fill="tozeroy", fillcolor=with_alpha(color, 0.12),
                               marker=dict(color=color, size=sizes, line=dict(color=SURFACE, width=1)), customdata=hover,
                               hovertemplate="%{x|%b %Y}<br>%{customdata}<extra></extra>", showlegend=False, connectgaps=False))
    fig.update_yaxes(title=MEASURE_LABELS[measure], exponentformat="B", rangemode="tozero", fixedrange=True)
    fig.update_xaxes(type="date", showgrid=False, **({"tickformat": "%b %Y"} if len(YEARS) > 1 else {"tickformat": "%b", "dtick": "M1"}))
    return apply_plotly_theme(fig, height=height)


country, measure, summary = view()
say("How much did the largest event of each month expose?", "One chart per measure, month by month. Each point is the largest single event of that month (hover for its name); values are never added across events.")
for item in MEASURE_NAMES:
    note(f"{MEASURE_LABELS[item]} · largest event of the month")
    show(fig_measure_timeseries(*largest_per_month(summary, item), item) if len(summary) else empty_figure("No events in this view.", height=200))
if "capital" in MEASURE_NAMES:
    note(f"Capital: {CAPITAL_UNIT_NOTE}. Values are shown as published; no currency is assumed.")

In [ ]:
#@title Functions: exceedance_curve, fig_exceedance
# How many events exposed at least X?
def exceedance_curve(summary, measure, by="hazard_group"):
    """For every group, how many events had a peak value of at least each threshold."""
    data = summary[pd.to_numeric(summary[f"{measure}_peak"], errors="coerce").gt(0)]
    rows = []
    for group, frame in data.groupby(by, sort=False):
        values = np.sort(frame[f"{measure}_peak"].to_numpy(dtype=float))[::-1]
        rows.extend({by: group, "threshold": value, "events": rank} for rank, value in enumerate(values, start=1))
    curve = pd.DataFrame(rows, columns=[by, "threshold", "events"])
    return curve.groupby([by, "threshold"], as_index=False, sort=False)["events"].max()  # events that share a value count together


def fig_exceedance(curve, measure, by="hazard_group", colors=None):
    """Step lines of the exceedance curve, one per hazard group (or country); both axes logarithmic."""
    if curve.empty:
        return empty_figure(f"No positive {MEASURE_NOUNS[measure]} values to draw.")
    order = [g for g in HAZARD_GROUP_ORDER if g in set(curve[by])] if by == "hazard_group" else list(dict.fromkeys(curve[by]))
    fig = go.Figure()
    for group in order:
        rows, label = curve[curve[by] == group].sort_values("threshold"), (country_name(group) if by == "country" else str(group))
        fig.add_scatter(x=rows["threshold"], y=rows["events"], mode="lines", line=dict(color=(colors or {}).get(group) or hazard_color(group), width=2, shape="vh"),
                        name=label, customdata=rows["threshold"].map(compact), hovertemplate=f"{label}: %{{y}} events with ≥ %{{customdata}}<extra></extra>")
    fig.update_xaxes(**log_axis(title=f"At least this many {MEASURE_NOUNS[measure]} (event peak)"))
    top = float(curve["events"].max())
    ticks = [value for value in (1, 3, 10, 30, 100, 300, 1000, 3000, 10000, 30000) if value <= top * 1.5]
    fig.update_yaxes(title="Number of events (log scale)", type="log", tickvals=ticks, ticktext=[compact(v) for v in ticks], range=[math.log10(0.8), math.log10(top * 1.4)])
    return apply_plotly_theme(fig, height=360)


country, measure, summary = view()
say(f"How many events exposed at least X {MEASURE_NOUNS[measure]}?", "Peak value per event; one line per hazard group; log scales.")
show(fig_exceedance(exceedance_curve(summary, measure), measure))

In [ ]:
#@title Functions: histogram_bins, fig_distribution
def histogram_bins(values):
    """Counts in readable 1-3-10 bins (1k–3k, 3k–10k, …), roughly half a decade each; empty bins at both ends are dropped."""
    low_decade, high_decade = math.floor(math.log10(values.min())), math.ceil(math.log10(values.max()))
    edges = np.array(sorted({base * 10.0 ** power for power in range(low_decade, high_decade + 1) for base in (1, 3)}))
    edges = edges[(edges <= values.max() * 3) & (edges * 3 >= values.min())]
    edges = np.append(edges, edges[-1] * (10 / 3 if str(int(edges[-1]))[0] == "3" else 3))
    counts, edges = np.histogram(values, bins=edges)
    labels = [f"{compact(low)}–{compact(high)}" for low, high in zip(edges[:-1], edges[1:], strict=True)]
    keep = np.flatnonzero(counts)
    return labels[keep.min(): keep.max() + 1], counts[keep.min(): keep.max() + 1]


def fig_distribution(summary, measure, kind="histogram"):
    """A histogram of event peaks, or their cumulative share (ECDF)."""
    data = positive(summary, measure)
    if data.empty:
        return empty_figure(f"No positive {MEASURE_NOUNS[measure]} values to draw.")
    values, fig = np.sort(data[f"{measure}_peak"].to_numpy(dtype=float)), go.Figure()
    if kind == "ecdf":
        fig.add_scatter(x=values, y=np.arange(1, len(values) + 1) / len(values), mode="lines", line=dict(color=PRIMARY, width=2, shape="hv"),
                        customdata=[compact(value) for value in values], hovertemplate="%{y:.0%} of events ≤ %{customdata}<extra></extra>", showlegend=False)
        fig.update_xaxes(**log_axis(title=MEASURE_LABELS[measure] + " (event peak)"))
        fig.update_yaxes(title="Share of events", tickformat=".0%", range=[0, 1.02])
    else:
        labels, counts = histogram_bins(values)
        fig.add_bar(x=labels, y=counts, marker=dict(color=PRIMARY), hovertemplate="%{x}: %{y} events<extra></extra>", showlegend=False)
        fig.update_xaxes(type="category", title=MEASURE_LABELS[measure] + " (event peak)", showgrid=False)
        fig.update_yaxes(title="Number of events", rangemode="tozero")
    return apply_plotly_theme(fig, height=320)


country, measure, summary = view()
say("How large are events?", f"Number of events by peak {MEASURE_NOUNS[measure]} exposed.")
kind_picker = widgets.ToggleButtons(options=[("Histogram", "histogram"), ("Cumulative share", "ecdf")], value="histogram")
interactive([kind_picker], lambda summary, measure, kind: show(fig_distribution(summary, measure, kind)), summary, measure)

In [ ]:
#@title Functions: fig_infrastructure
def fig_infrastructure(summary, measure, n=8):
    """The events that exposed the most of one infrastructure measure (schools or hospitals), on their own axis."""
    data = positive(summary, measure)
    if data.empty:
        return empty_figure(f"No event had {MEASURE_NOUNS[measure]} exposed.", height=200)
    column = f"{measure}_peak"
    top = data.nlargest(n, column).iloc[::-1]
    keys = list(top["family_key"].astype(str))
    labels = [f"{short(title, 34)} · {date_text(date)}" for title, date in zip(top["title"], top["event_date"], strict=True)]
    fig = go.Figure(go.Bar(x=top[column], y=keys, orientation="h", marker=dict(color=[hazard_color(group) for group in top["hazard_group"]]), text=top[column].map(compact),
                           textposition="outside", textfont=dict(color=INK_SECONDARY, size=11), cliponaxis=False, customdata=hover_rows(top, measure), hovertemplate=EVENT_HOVER, showlegend=False))
    fig.update_yaxes(type="category", categoryorder="array", categoryarray=keys, tickvals=keys, ticktext=labels, showgrid=False, tickfont=dict(size=10.5))
    fig.update_xaxes(title=MEASURE_LABELS[measure] + " (peak)", exponentformat="B", rangemode="tozero", range=[0, float(top[column].max()) * 1.2])
    return apply_plotly_theme(fig, height=60 + 28 * len(top))


country, measure, summary = view()
for item in [name for name in ("schools", "hospitals") if name in MEASURE_NAMES]:
    exposing = int(pd.to_numeric(summary[f"{item}_peak"], errors="coerce").gt(0).sum()) if len(summary) else 0
    say(f"Events exposing the most {item}", f"{exposing} of {len(summary)} events exposed at least one.")
    show(fig_infrastructure(summary, item))

## 12. One event in detail

Pick an event below. Only this section is redrawn when you change the picker, and **no API call** is made for the exposure values: they are already in memory.
The one exception is the **PDC alert area**: for the selected event only, the notebook fetches the "Maps" file PDC advertises for it, validates it, and shows its access status.
Nothing is invented if the file cannot be read; the map then shows the PDC point only.

The section shows: the event card, the peak and latest value of every measure, the **PDC alert area** on a map (the outline PDC used for its alert, **not** a verified impact footprint),
the **age profile** (only when *Age groups* was chosen in step 3), the **snapshot timeline** (when PDC published each update) and how the chosen measure **changed over time**.

`event_options` lists the events for the picker. `event_card` prints the card, and `measure_cards` the values of every measure.

In [ ]:
#@title Functions: event_options, chip, event_card
def event_options(summary, measure):
    """The events for the picker, largest peak first, as (label, family key)."""
    ordered = summary.sort_values(f"{measure}_peak", ascending=False, na_position="last")
    return [(f"{short(row.title, 60)} · {date_text(row.event_date)} · {compact(getattr(row, f'{measure}_peak'))}", row.family_key) for row in ordered.itertuples()]


def chip(text, background, color="#FFFFFF"):
    return f"<span style='background:{background};color:{color};border-radius:999px;padding:1px 9px;font-size:.78rem;font-weight:600;margin-right:4px'>{escape(text)}</span>"


def event_card(row):
    """The event's name, hazard, alert level, countries, dates and caveat as a small card."""
    caveat = row["caveat"] if isinstance(row["caveat"], str) else ""
    report = row["report_url"] if isinstance(row["report_url"], str) else ""
    kind = row["pdc_hazard_type"] if isinstance(row["pdc_hazard_type"], str) else row["hazard_group"]
    chips = chip(row["hazard_group"], hazard_color(row["hazard_group"]))
    if isinstance(row["alert_level_max"], str):
        chips += chip(f"PDC {row['alert_level_max'].title()}", ALERT_COLORS.get(row["alert_level_max"], "#98A3B3"))
    chips += chip(f"{row['n_countries']} countries", "rgba(128,128,128,.25)", "inherit") if row["multi_country"] else ""
    chips += chip("Caveat", "#FBF1DD", "#7A5200") if caveat else ""
    updates = int(row["snapshot_count"])
    lines = [f"{kind} · started {date_text(row['event_date'])} · {updates} PDC update{'s' if updates != 1 else ''}",
             "Countries: " + ", ".join(country_name(code) for code in row["countries"]),
             f"PDC updates from {date_text(row['first_snapshot_time'])} to {date_text(row['last_snapshot_time'])}"]
    link = f"<a href='{escape(report)}' target='_blank' rel='noopener'>Open the PDC event report</a>" if report else ""
    display(HTML(f"<div style='border:1px solid rgba(128,128,128,.35);border-radius:12px;padding:12px 16px;margin:8px 0'>"
                 f"<div style='font-size:1.08rem;font-weight:700'>{escape(str(row['title']))}</div><div style='margin:4px 0'>{chips}</div>"
                 + "".join(f"<div style='opacity:.8;font-size:.85rem'>{escape(line)}</div>" for line in lines)
                 + (f"<div style='font-size:.82rem;margin-top:6px'>{escape(caveat)}</div>" if caveat else "") + f"<div style='margin-top:6px'>{link}</div></div>"))

In [ ]:
#@title Functions: measure_cards
def measure_cards(row):
    """Peak, latest and number of changes of each measure. A missing, conflicting or unretrieved value is named; it is not shown as zero."""
    words = {"missing": "Not published", "unavailable": "No numeric value", "conflicting": "Conflicting values", "not_retrieved": "Not retrieved"}
    items = []
    for item in MEASURE_NAMES:
        status, label = row[f"{item}_status"], MEASURE_LABELS[item].replace(" (unit not stated)", "")
        if status in ("present_positive", "present_zero"):
            items.append((label, compact(row[f"{item}_peak"]), f"Latest {compact(row[f'{item}_latest'])} · {int(row[f'{item}_changes'])} changes"))
        else:
            items.append((label, "–", words.get(status, status)))
    return items

In [ ]:
#@title Functions: feature_collection, area_layers
def feature_collection(geometries):
    return {"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {}, "geometry": geometry} for geometry in geometries]}


def area_layers(footprint, color):
    """Map layers for the alert-area versions, and the slider steps that switch between them."""
    drawn = [display_geometry(area.geometry, footprint.display_step) for area in footprint.areas]
    layers = [{"source": feature_collection(drawn), "type": "line", "color": color, "opacity": 0.35, "line": {"width": 0.6}, "below": "traces"}] if len(drawn) > 1 else []
    layers.append({"source": feature_collection(drawn[-1:]), "type": "fill", "color": color, "opacity": 0.28, "fill": {"outlinecolor": color}, "below": "traces"})
    steps = []
    for index, (area, geometry) in enumerate(zip(footprint.areas, drawn, strict=True)):
        label = area.created.strftime("%d %b %H:%M") if area.created is not None else f"v{index + 1}"
        label += f" · {compact(area.area_km2)} km²" if area.area_km2 is not None else ""
        steps.append({"label": label, "method": "relayout", "args": [{"map.layers[1].source": feature_collection([geometry])}]})
    return layers, steps

In [ ]:
#@title Functions: fig_alert_area_map
def fig_alert_area_map(row, footprint, height=460):
    """The PDC point and the alert-area outline of one event (latest version filled, earlier versions faint, a slider to step through them)."""
    color, fig = hazard_color(str(row["hazard_group"])), go.Figure()
    if row["valid_point"]:
        fig.add_trace(go.Scattermap(lat=[row["latitude"]], lon=[row["longitude"]], mode="markers", marker=dict(size=13, color=SURFACE), hoverinfo="skip", showlegend=False))
        fig.add_trace(go.Scattermap(lat=[row["latitude"]], lon=[row["longitude"]], mode="markers", marker=dict(size=9, color=INK), name="PDC event point",
                                    hovertemplate=f"PDC event point<br>{short(row['title'])}<extra></extra>"))
    for line in footprint.lines:  # a cyclone track, when the file has one
        xs, ys = zip(*[point[:2] for point in line["geometry"]["coordinates"]], strict=True)
        fig.add_trace(go.Scattermap(lon=list(xs), lat=list(ys), mode="lines", line=dict(color=color, width=2), hoverinfo="skip", showlegend=False))
    if footprint.bbox is not None:
        center, zoom, _ = fit_view(np.array([footprint.bbox[0], footprint.bbox[2]]), np.array([footprint.bbox[1], footprint.bbox[3]]), trim=False)
    elif row["valid_point"]:
        center, zoom = {"lon": float(row["longitude"]), "lat": float(row["latitude"])}, 6.0
    else:
        return empty_figure("No PDC point or alert area to map.", height=240)
    layers, steps = area_layers(footprint, color) if footprint.areas else ([], [])
    fig.update_layout(map_layers=layers, showlegend=False, map=dict(style="carto-positron", center=center, zoom=zoom), margin=dict(l=0, r=0, t=0, b=70 if len(steps) > 1 else 0))
    if len(steps) > 1:
        fig.update_layout(sliders=[{"active": len(steps) - 1, "steps": steps, "x": 0.02, "len": 0.96, "y": 0, "yanchor": "top", "pad": {"t": 6, "b": 4},
                                    "currentvalue": {"prefix": "PDC alert area at ", "font": {"size": 12, "color": INK}}, "font": {"size": 10, "color": "rgba(0,0,0,0)"},
                                    "tickcolor": "#C9CED6", "bgcolor": "#EEF0F3", "activebgcolor": PRIMARY, "bordercolor": "#E2E5EA"}])
    return apply_plotly_theme(fig, height=height)

In [ ]:
#@title Functions: alert_area_note, draw_alert_area
ALERT_AREAS = {}  # fetched alert-area files by URL, so a picker change does not fetch the same file twice


def alert_area_note(footprint):
    """The access status of the alert-area file, and what the outline means."""
    if footprint.status != "available":
        return f"Alert area not shown: {footprint.message} The map shows the PDC point only."
    latest = footprint.latest
    area = f"{compact(latest.area_km2)} km²" if latest is not None and latest.area_km2 is not None else "not computed"
    inside = {True: "inside", False: "outside", None: "not checked against"}[footprint.point_inside]
    return f"{len(footprint.areas)} PDC alert-area version(s); latest area {area}; the PDC point is {inside} the latest area. " + " ".join(footprint.notes) + " " + ALERT_AREA_NOTE


def draw_alert_area(row):
    """Fetch (once) and draw the selected event's alert area, with its access status."""
    say("PDC alert area", HAZARD_MAP_NOTES.get(row["hazard_group"], ""))
    point = (float(row["longitude"]), float(row["latitude"])) if row["valid_point"] else None
    url = row["footprint_url"] if isinstance(row["footprint_url"], str) else None
    if url not in ALERT_AREAS:
        ALERT_AREAS[url] = fetch_alert_areas(url, allowed_hosts=config.footprint_hosts, point=point)
    show(fig_alert_area_map(row, ALERT_AREAS[url]))
    note(alert_area_note(ALERT_AREAS[url]))

### Who was exposed, by age?
PDC splits the people it counts into 14 age bands. The bands are shown at the snapshot of the event's **peak** `people` value. **`people` is a separate value and is never added to the bands**:
the sum of the bands is only compared with it, as a consistency check. PDC applies one national age structure to every event of a country, so the *shares* are the same for every event; only the counts carry information.

In [ ]:
#@title Functions: age_snapshot, age_profile
AGE_SHORT = {"children_0_4": "0–4", "children_5_9": "5–9", "children_10_14": "10–14", "children_15_19": "15–19", "adult_20_24": "20–24", "adult_25_29": "25–29",
             "adult_30_34": "30–34", "adult_35_39": "35–39", "adult_40_44": "40–44", "adult_45_49": "45–49", "adult_50_54": "50–54", "adult_55_59": "55–59",
             "adult_60_64": "60–64", "elderly": "65+"}


def age_snapshot(data):
    """The snapshot time to show: the one of the peak `people` value if it has age bands, else the latest time with bands."""
    bands, people = data[data["impact_category"].isin(AGE_BAND_CATEGORIES)], data[data["impact_category"] == "people"]
    band_times = set(bands["time"])
    if not people.empty:
        peak_times = people[people["value"] == people["value"].max()].sort_values("time")["time"]
        candidates = [time for time in peak_times if time in band_times]
        if candidates:
            return candidates[-1]
    return max(band_times)


def age_profile(impacts, family_key):
    """The age bands of one event at one snapshot, and the reconciliation with the separate `people` value."""
    data = impacts[(impacts["family_key"] == family_key) & impacts["impact_category"].isin(("people", *AGE_BAND_CATEGORIES))].copy()
    data["value"], data["time"] = pd.to_numeric(data["numeric_value"], errors="coerce"), data["snapshot_time"].fillna(data["item_datetime"]).fillna("")
    data = data.dropna(subset=["value"])
    if not data["impact_category"].isin(AGE_BAND_CATEGORIES).any():
        return pd.DataFrame(columns=["band", "value", "share"]), {}
    time = age_snapshot(data)
    at_time = data[(data["time"] == time) & data["impact_category"].isin(AGE_BAND_CATEGORIES)].drop_duplicates("impact_category", keep="last")
    at_time = at_time.assign(order=at_time["impact_category"].map({c: i for i, c in enumerate(AGE_BAND_CATEGORIES)})).sort_values("order")
    profile = pd.DataFrame({"band": at_time["impact_category"].map(AGE_SHORT).to_numpy(), "value": at_time["value"].to_numpy()})
    profile["share"] = profile["value"] / profile["value"].sum() if profile["value"].sum() else np.nan
    people = data[(data["impact_category"] == "people") & (data["time"] == time)]["value"]
    people_value = float(people.iloc[-1]) if not people.empty else None
    band_sum = float(profile["value"].sum())
    return profile, {"time": time, "people": people_value, "band_sum": band_sum, "difference": band_sum - people_value if people_value is not None else None,
                     "difference_pct": (band_sum - people_value) * 100 / people_value if people_value else None}

In [ ]:
#@title Functions: fig_age_profile, fig_age_reconciliation, draw_age
def fig_age_profile(profile):
    """Horizontal bars of the age bands, youngest first. `people` is not a band and is not drawn here."""
    fig = go.Figure(go.Bar(x=profile["value"], y=profile["band"], orientation="h", marker=dict(color=PRIMARY), text=profile["value"].map(compact), textposition="outside",
                           cliponaxis=False, customdata=(profile["share"] * 100).round(1), hovertemplate="Age %{y}: %{x:,.0f} (%{customdata}% of the age-band total)<extra></extra>"))
    fig.update_yaxes(autorange="reversed", type="category", title="Age band (years)", showgrid=False)
    fig.update_xaxes(title="Exposed in the age band", rangemode="tozero", exponentformat="B", range=[0, float(profile["value"].max()) * 1.15])
    return apply_plotly_theme(fig, height=70 + 26 * len(profile))


def fig_age_reconciliation(info):
    """Two bars: the sum of the age bands and the separate `people` value. Only a consistency check: neither replaces the other."""
    labels, values = ["Sum of the age bands", "People (separate PDC value)"], [info["band_sum"], info["people"]]
    fig = go.Figure(go.Bar(x=values, y=labels, orientation="h", marker=dict(color=[PRIMARY, CONTEXT]), text=[compact(v) for v in values], textposition="outside",
                           cliponaxis=False, hovertemplate="%{y}: %{x:,.0f}<extra></extra>"))
    fig.add_annotation(text=f"Difference {info['difference']:+,.0f} ({info['difference_pct']:+.1f}%)", xref="paper", yref="paper", x=1, y=1.12, showarrow=False, font=dict(size=12, color=INK_SECONDARY))
    fig.update_yaxes(autorange="reversed", showgrid=False)
    fig.update_xaxes(rangemode="tozero", exponentformat="B", range=[0, max(values) * 1.2])
    return apply_plotly_theme(fig, height=170)


def draw_age(country, family_key):
    """The age profile and its reconciliation, when age groups were retrieved."""
    if not HAS_AGE_BANDS:
        return note("Age groups were not chosen in step 3, so no age bands were retrieved. Tick 'Age groups', then run again from step 4.")
    say("Who was exposed, by age?", "Age bands at the snapshot of the event's peak people value.")
    profile, info = age_profile(TABLES[country]["impacts"], family_key)
    if profile.empty:
        return note("PDC published no age bands for this event.")
    show(fig_age_profile(profile))
    if info.get("people") is not None:
        show(fig_age_reconciliation(info))
    else:
        note("No separate `people` value at this snapshot, so the age bands cannot be compared with it.")
    note("PDC applies one national age structure to every event of a country, so age shares do not differ between events; the counts show scale.")

### When did PDC publish each update, and how did the value change?
An event's `datetime` stays constant, but PDC re-publishes it as it updates, each time with a new exposure estimate.
**The snapshot timeline** shows the time of every update (read from the item ID) against the event's start and end.
**The change chart** shows the chosen measure's value at each snapshot as a step line, with a marker only where the value changed.
These are historical **PDC exposure snapshot values**, not observed impact reports; two different values for one time are marked, not resolved; nothing is forecast.

In [ ]:
#@title Functions: snapshot_history, history_sentence
def snapshot_history(impacts, family_key, category):
    """One row per (snapshot time, value) of one event and category; flags the first value and every change, and times with two different values."""
    data = impacts[(impacts["family_key"] == family_key) & (impacts["impact_category"] == category)].copy()
    data["time"] = pd.to_datetime(data["snapshot_time"], errors="coerce", utc=True, format="mixed")
    data["value"] = pd.to_numeric(data["numeric_value"], errors="coerce")
    history = data.dropna(subset=["time", "value"]).drop_duplicates(["time", "value"]).sort_values(["time", "value"]).reset_index(drop=True)[["time", "value"]]
    history["conflict"] = history.duplicated("time", keep=False)
    clean = history[~history["conflict"]]
    history["change"] = False
    history.loc[clean.index, "change"] = clean["value"].diff().fillna(1).ne(0)
    return history


def history_sentence(history, measure):
    """A plain-words summary of the history (never a forecast)."""
    clean = history[~history["conflict"]]
    if len(clean) < 2:
        return f"Only {len(clean)} PDC snapshot value for {MEASURE_NOUNS[measure]}: not enough history to describe a change."
    first, last = clean["value"].iloc[0], clean["value"].iloc[-1]
    state = "stayed the same" if clean["value"].nunique() == 1 else "increased" if last > first else "decreased" if last < first else "changed"
    return f"The {MEASURE_NOUNS[measure]} value {state} across {len(clean)} PDC snapshots: {first:,.0f} to {last:,.0f}. Historical PDC exposure snapshot values; not an observed impact report; no forecast."

In [ ]:
#@title Functions: fig_snapshot_timeline, fig_change_timeline
def fig_snapshot_timeline(row, snapshot_times):
    """When PDC published each snapshot of the event, against the event's start and end (UTC)."""
    color, fig = hazard_color(row["hazard_group"]), go.Figure()
    if pd.notna(row["start"]) and pd.notna(row["end"]):
        fig.add_shape(type="rect", x0=row["start"], x1=row["end"], y0=0.15, y1=0.85, fillcolor=with_alpha(color, 0.14), line=dict(width=0))
    fig.add_scatter(x=snapshot_times, y=[0.5] * len(snapshot_times), mode="markers", name="PDC snapshot", marker=dict(size=9, color=PRIMARY, line=dict(color=SURFACE, width=1)),
                    hovertemplate="Snapshot %{x|%d %b %Y %H:%M} UTC<extra></extra>")
    fig.add_scatter(x=[row["event_date"]], y=[0.5], mode="markers", name="Event datetime", marker=dict(symbol="diamond", size=12, color=INK),
                    hovertemplate="Event datetime %{x|%d %b %Y %H:%M} UTC<extra></extra>")
    fig.update_yaxes(visible=False, range=[0, 1])
    fig.update_xaxes(title="Exposure snapshot time (UTC), read from the PDC item ID; shaded = event start to end")
    return apply_plotly_theme(fig, height=170)


def fig_change_timeline(history, measure):
    """The measure's value at each PDC snapshot as a step line. Markers only where the value changed; red crosses mark two values at one time."""
    clean, color = history[~history["conflict"]], MEASURE_COLORS[measure]
    fig = go.Figure(go.Scatter(x=clean["time"], y=clean["value"], mode="lines", line=dict(color=color, width=2, shape="hv"), hoverinfo="skip", showlegend=False))
    changed = clean[clean["change"]]
    fig.add_scatter(x=changed["time"], y=changed["value"], mode="markers", name="value changed", marker=dict(color=color, size=9, line=dict(color=SURFACE, width=1)),
                    hovertemplate="%{x|%d %b %Y %H:%M} UTC<br>%{y:,.0f}<extra></extra>")
    conflicts = history[history["conflict"]]
    if len(conflicts):
        fig.add_scatter(x=conflicts["time"], y=conflicts["value"], mode="markers", name="two values at one time", marker=dict(symbol="x", size=10, color="#B8312F"))
    fig.update_yaxes(title=MEASURE_LABELS[measure], rangemode="tozero", exponentformat="B")
    fig.update_xaxes(title="Exposure snapshot time (UTC)")
    return apply_plotly_theme(fig, height=300)

In [ ]:
#@title Functions: draw_history, draw_event
def draw_history(country, row, measure):
    """The snapshot timeline and the change chart of the chosen measure."""
    say("PDC snapshots over time", "Each dot is one PDC update of this event.")
    events = TABLES[country]["events"]
    times = pd.to_datetime(events.loc[events["family_key"] == row["family_key"], "snapshot_time"], errors="coerce", utc=True, format="mixed").dropna()
    if len(times):
        show(fig_snapshot_timeline(row, times))
    else:
        note("No snapshot time could be read from the item IDs of this event.")
    history = snapshot_history(TABLES[country]["impacts"], row["family_key"], MEASURE_CATEGORY[measure])
    if history.empty:
        return note(f"No numeric {MEASURE_NOUNS[measure]} snapshot values with a readable time for this event.")
    show(fig_change_timeline(history, measure))
    note(history_sentence(history, measure))


def draw_event(country, measure, family_key):
    """Everything about one event; this is what the picker redraws."""
    row = SUMMARIES[country].set_index("family_key", drop=False).loc[family_key]
    event_card(row)
    cards(measure_cards(row))
    if "capital" in MEASURE_NAMES and isinstance(row["description_currency_quote"], str):
        note(f"{CAPITAL_UNIT_NOTE}. PDC's description says: “{row['description_currency_quote']}”")
    draw_alert_area(row)
    draw_age(country, family_key)
    draw_history(country, row, measure)


country, measure, summary = view()
if summary.empty:
    note("No events in this view.")
else:
    event_picker = widgets.Dropdown(options=event_options(summary, measure), description="Event", style={"description_width": "50px"}, layout=widgets.Layout(width="95%"))
    interactive([event_picker], draw_event, country, measure)

## 13. Compare countries

This section appears only when **two or more countries** were retrieved. It shows where the countries' events are, how many events each had per period, which hazards dominate and how the size of events compares.

**Country totals overlap and are not added.** An event that lists several of the selected countries is counted in each of them. Counts are distinct PDC events, never summed exposure.
PDC exposure values belong to the country they were retrieved for (a shared event has a different value in each), so the combined map has **no exposure values**: its dots all have one size.

In [ ]:
#@title Functions: compared_summaries, combined_events, country_counts
COMPARE = len(COUNTRIES) > 1
COUNTRY_COLORS = ("#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7", "#e34948", "#eda100", "#e87ba4", "#008300")
COUNTRY_COUNT_NOTE = ("An event that lists several of the selected countries is counted in each of them, so country counts overlap and must not be added. "
                      "Counts are distinct PDC events, never summed exposure.")
if not COMPARE:
    note("Only one country was retrieved. Choose two or more countries in step 3 and run again from step 4 to compare them here.")


def compared_summaries():
    """The events of every country, with tsunami bulletins hidden unless chosen in step 7."""
    return {code: (s if bulletin_toggle.value or s.empty else s[~is_bulletin(s)]).reset_index(drop=True) for code, s in SUMMARIES.items()}


def combined_events(summaries):
    """Every event of the compared countries once. An event shared by two countries has a different value in each, so no exposure value is kept."""
    parts = [s for s in summaries.values() if len(s)]
    combined = pd.concat(parts, ignore_index=True).drop_duplicates("family_key").reset_index(drop=True) if parts else pd.DataFrame(columns=SUMMARY_COLUMNS)
    for measure in SUMMARY_MEASURES:
        combined[f"{measure}_peak"] = combined[f"{measure}_latest"] = np.nan
    combined["country"] = None
    return combined


def country_counts(combined, countries):
    """Distinct events per country. An event listing several countries is counted in each of them, so the counts overlap and are not added."""
    rows = combined[["family_key", "countries", "hazard_group", "snapshot_count", "multi_country"]].explode("countries").dropna(subset=["countries"])
    rows = rows.rename(columns={"countries": "country_code"}).drop_duplicates(["family_key", "country_code"])
    rows = rows[rows["country_code"].isin(countries)]
    grouped = rows.groupby("country_code")
    counts = pd.DataFrame({"event_families": grouped["family_key"].nunique(), "event_snapshots": grouped["snapshot_count"].sum().astype(int),
                           "multi_country_events": grouped["multi_country"].sum().astype(int)})
    by_hazard = rows.groupby(["country_code", "hazard_group"])["family_key"].nunique().rename("events").reset_index()
    by_hazard = by_hazard.sort_values(["country_code", "events", "hazard_group"], ascending=[True, False, True], kind="stable")
    counts["top_hazards"] = by_hazard.groupby("country_code").apply(
        lambda f: ", ".join(f"{group} ({events})" for group, events in zip(f["hazard_group"].head(3), f["events"].head(3), strict=True)), include_groups=False)
    return counts.reset_index().sort_values(["event_families", "country_code"], ascending=[False, True], kind="stable").reset_index(drop=True)

In [ ]:
#@title Functions: nice_scale, fig_choropleth
# Where are the selected countries' events?
def nice_scale(maximum, ticks=5):
    """A round upper bound and tick step for a colour bar that starts at 0 (for example 358 gives 400 in steps of 100)."""
    if not maximum or maximum <= 0 or not math.isfinite(maximum):
        return 1.0, 1.0
    raw = maximum / (ticks - 1)
    magnitude = 10 ** math.floor(math.log10(raw))
    step = max(next(base * magnitude for base in (1, 2, 2.5, 5, 10) if base * magnitude >= raw), 1.0)  # counts are whole numbers
    return math.ceil(maximum / step) * step, step


def fig_choropleth(counts, height=380):
    """The selected countries coloured by distinct events (never by summed exposure)."""
    zmax, step = nice_scale(float(counts["event_families"].max()))
    details = np.column_stack([counts["country_code"], counts["country_code"].map(country_name), counts["event_families"], counts["event_snapshots"],
                               counts["multi_country_events"], counts["top_hazards"]])
    fig = go.Figure(go.Choropleth(
        locations=counts["country_code"], z=counts["event_families"].astype(float), zmin=0, zmax=zmax, locationmode="ISO-3", colorscale=sequential_scale(),
        marker_line_color=SURFACE, marker_line_width=0.5, customdata=details,
        hovertemplate=("<b>%{customdata[1]}</b> · " + period_label + "<br>%{customdata[2]} distinct events · %{customdata[3]} PDC updates"
                       "<br>%{customdata[4]} also list other countries<br>%{customdata[5]}<extra></extra>"),
        colorbar=dict(title=dict(text="Distinct events", font=dict(size=11)), thickness=12, len=0.75, tickmode="linear", tick0=0, dtick=step, tickformat=",d", ticks="outside", ticklen=4)))
    fig.update_geos(projection_type="natural earth", showframe=False, showcoastlines=False, landcolor="#EEF0F3", showland=True, showcountries=True,
                    countrycolor=SURFACE, bgcolor="rgba(0,0,0,0)", fitbounds="locations")
    fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))
    return apply_plotly_theme(fig, height=height)


country, measure, summary = view()
COMPARE_READY = False
if COMPARE:
    summaries = compared_summaries()
    combined = combined_events(summaries)
    counts = country_counts(combined, COUNTRIES) if len(combined) else pd.DataFrame()
    COMPARE_READY = len(combined) > 0
    if not COMPARE_READY:
        note("The selected countries have no events in this view.")
if COMPARE_READY:
    say("Where are the selected countries' events?", "Distinct PDC events per selected country. Darker = more events.")
    show(fig_choropleth(counts))
    note(COUNTRY_COUNT_NOTE)

In [ ]:
#@title Functions: draw_combined
# All selected countries on one map
def draw_combined(combined, measure, layer):
    """The combined event map, with its caption."""
    show(fig_event_map(combined, measure, layer=layer, fit_all=True, uniform_size=9, zoom_offset=-0.6, height=520))
    note("Dots have one size here: PDC exposure values are per country, so a shared event has a different value in each. Each country's own map (step 9) shows sizes and values."
         + (" " + CLUSTER_NOTE if layer == "clusters" else ""))


if COMPARE_READY:
    say("All selected countries on one map", f"{len(combined):,} distinct events; an event shared by several selected countries appears once.")
    combined_layer = widgets.ToggleButtons(options=[("Events", "points"), ("Clusters", "clusters")], value="clusters")
    interactive([combined_layer], draw_combined, combined, measure)

In [ ]:
#@title Functions: period_counts, fig_compare_per_period
# Events per period in each country
def period_counts(summary):
    """Distinct events per period (year, or month when one year is loaded) and hazard group."""
    data = summary.dropna(subset=["year", "month"])
    period = data["year"].astype(int).astype(str) if len(YEARS) > 1 else data["month"].astype(int).map(lambda month: MONTH_NAMES[month - 1])
    return data.assign(period=period).groupby(["period", "hazard_group"])["family_key"].nunique()


def fig_compare_per_period(summaries):
    """Small multiples: one panel per country of distinct events per period, stacked by hazard group."""
    countries, seen = list(summaries), set()
    categories = [str(year) for year in YEARS] if len(YEARS) > 1 else list(MONTH_NAMES)
    fig = make_subplots(rows=1, cols=len(countries), shared_yaxes=True, subplot_titles=[country_name(c) for c in countries], horizontal_spacing=0.03)
    for column, country in enumerate(countries, start=1):
        counts = period_counts(summaries[country]) if len(summaries[country]) else pd.Series(dtype=int)
        for group in groups_present(summaries[country]):
            fig.add_bar(x=categories, y=[int(counts.get((period, group), 0)) for period in categories], name=group, legendgroup=group, showlegend=group not in seen,
                        marker=dict(color=hazard_color(group), line=dict(color=SURFACE, width=1)), row=1, col=column,
                        hovertemplate=f"{country_name(country)} %{{x}} · {group}: %{{y}} events<extra></extra>")
            seen.add(group)
    fig.update_layout(barmode="stack")
    fig.update_xaxes(type="category", showgrid=False, tickangle=0)
    fig.update_yaxes(title_text="Distinct events", row=1, col=1)
    for annotation in fig.layout.annotations:
        annotation.font = dict(size=13, color=INK)
    fig = apply_plotly_theme(fig, height=360)
    return fig.update_layout(legend=dict(orientation="h", yanchor="top", y=-0.12, xanchor="left", x=0, traceorder="normal"), margin=dict(t=28, b=8))


if COMPARE_READY:
    say("Events per period in each country", "Distinct events, stacked by hazard, on a shared scale.")
    show(fig_compare_per_period(summaries))

In [ ]:
#@title Functions: hazard_mix, fig_hazard_mix
# Which hazards dominate in each country, and how big are events?
def hazard_mix(summaries, measure):
    """Distinct events and the median peak for every country and hazard group."""
    rows = [(country, group, frame["family_key"].nunique(), pd.to_numeric(frame[f"{measure}_peak"], errors="coerce").median())
            for country, data in summaries.items() for group, frame in data.groupby("hazard_group")]
    return pd.DataFrame(rows, columns=["country", "group", "events", "median"])


def fig_hazard_mix(table, countries, measure):
    """A country × hazard grid of distinct events; the hover shows the median peak."""
    if table.empty:
        return empty_figure("No events in the loaded countries.")
    groups = [group for group in HAZARD_GROUP_ORDER if group in set(table["group"])]

    def cell(country, group):
        return table[(table["country"] == country) & (table["group"] == group)]

    z = [[int(cell(c, g)["events"].sum()) for g in groups] for c in countries]
    medians = [[compact(cell(c, g)["median"].iloc[0]) if len(cell(c, g)) else "–" for g in groups] for c in countries]
    fig = go.Figure(go.Heatmap(z=z, x=groups, y=[country_name(c) for c in countries], colorscale=sequential_scale(), zmin=0, zmax=max(max(row) for row in z) or 1, xgap=3, ygap=3,
                               showscale=False, text=[[str(v) if v else "" for v in row] for row in z], texttemplate="%{text}", textfont=dict(size=12), customdata=medians,
                               hovertemplate="%{y} · %{x}: %{z} events<br>Median peak " + MEASURE_NOUNS[measure] + ": %{customdata}<extra></extra>"))
    fig.update_yaxes(autorange="reversed", type="category", showgrid=False, fixedrange=True)
    fig.update_xaxes(side="top", type="category", showgrid=False, tickangle=0, fixedrange=True)
    return apply_plotly_theme(fig, height=90 + 46 * len(countries))


if COMPARE_READY:
    say("Which hazards dominate in each country?", "Distinct events per hazard group. Hover for the median peak value.")
    show(fig_hazard_mix(hazard_mix(summaries, measure), list(summaries), measure))
    say(f"How many events exposed at least X {MEASURE_NOUNS[measure]}?", "One line per country; peak value per event; log scales.")
    everything = pd.concat([s for s in summaries.values() if len(s)], ignore_index=True)
    show(fig_exceedance(exceedance_curve(everything, measure, by="country"), measure, by="country", colors=dict(zip(summaries, COUNTRY_COLORS))))

In [ ]:
#@title Functions: compare_table
def compare_table(summaries, counts, measure):
    """One row per country: events, median and largest peak, events at PDC 'Warning', events shared with other countries and the main hazards."""
    name, top = MEASURE_SHORT[measure].lower(), counts.set_index("country_code")["top_hazards"]
    rows = []
    for country, data in summaries.items():
        values = pd.to_numeric(data[f"{measure}_peak"], errors="coerce")
        rows.append({"Country": country_name(country), "Events": len(data), f"Median peak {name}": values[values > 0].median(), f"Largest peak {name}": values.max(),
                     "Warning-level events": int((data["alert_level_max"] == "WARNING").sum()), "Also list other countries": int(data["multi_country"].sum()),
                     "Main hazards": top.get(country, "–")})
    return pd.DataFrame(rows)


if COMPARE_READY:
    say("Country summary", "Exposure values are never added across countries or events. An event shared by several countries appears in each.")
    table = compare_table(summaries, counts, measure)
    display(table.style.format({column: "{:,.0f}" for column in table.columns if column.startswith(("Median", "Largest"))}, na_rep="–").hide(axis="index"))

## 14. Data quality

**How complete and comparable is the evidence?** This section shows what was retrieved and what was not: coverage per year, whether each value is present, how the retrieval went,
which events carry a caveat, how events were linked, and what the geometry can be used for. Zero, missing, unavailable and conflicting values are kept apart throughout.

In [ ]:
#@title Functions: coverage_by_year, fig_coverage
def coverage_by_year(events):
    """PDC event snapshots and distinct events retrieved per year: separates changes in coverage from hazard trends."""
    year = pd.to_datetime(events["event_datetime"], errors="coerce", utc=True, format="mixed").dt.year
    grouped = events.assign(year=year).groupby("year").agg(event_snapshots=("event_item_id", "nunique"), event_families=("family_key", "nunique"))
    return grouped.reindex(list(YEARS), fill_value=0).rename_axis("year").reset_index()


def fig_coverage(coverage):
    """Grouped bars of PDC updates (snapshots) and distinct events per year."""
    fig = go.Figure()
    fig.add_bar(x=coverage["year"].astype(str), y=coverage["event_snapshots"], name="PDC updates (event snapshots)", marker=dict(color=CONTEXT), hovertemplate="%{x}: %{y} event snapshots<extra></extra>")
    fig.add_bar(x=coverage["year"].astype(str), y=coverage["event_families"], name="Distinct events", marker=dict(color=PRIMARY), hovertemplate="%{x}: %{y} distinct events<extra></extra>")
    fig.update_layout(barmode="group")
    fig.update_xaxes(type="category", showgrid=False)
    fig.update_yaxes(title="Count", rangemode="tozero")
    return apply_plotly_theme(fig, height=260)


country, measure, summary = view()
incomplete_banner()
say("How much PDC data exists per year?", f"{country_name(country)}: event updates and distinct events retrieved per year.")
show(fig_coverage(coverage_by_year(TABLES[country]["events"])))
note("Montandon's PDC coverage differs strongly between years, so a change from year to year can reflect coverage, not hazard activity.")

In [ ]:
#@title Functions: status_table
STATUS_WORDS = {"present_positive": "Value > 0", "present_zero": "Zero", "missing": "Not published", "unavailable": "No numeric value",
                "conflicting": "Conflicting", "not_retrieved": "Not retrieved"}


def status_table(summary):
    """For every measure: how many events have a value above zero, a zero, no published value, no numeric value, or conflicting values."""
    table = pd.DataFrame({MEASURE_SHORT[m]: summary[f"{m}_status"].map(STATUS_WORDS).value_counts() for m in MEASURE_NAMES}).fillna(0).astype(int).T
    return table[[word for word in STATUS_WORDS.values() if word in table.columns]]


country, measure, summary = view()
say("Is each value present?", "Per event and measure: a value above zero, a zero, not published, no numeric value, or conflicting values (two values for one snapshot time). Tsunami bulletins are included.")
display(status_table(SUMMARIES[country]))
note("Zero means PDC reported 0. Not published means PDC published no item for that event. They are different things and are never merged.")

In [ ]:
#@title Functions: retrieval_table, month_table
def retrieval_table(log):
    """Per country and collection: items, pages, retried requests, split windows, windows that could not be read and repeated item IDs."""
    table = log.groupby(["country", "collection"]).agg(
        items=("items", "sum"), pages=("pages", "sum"), retries=("retries", "sum"), split_windows=("note", lambda notes: int(notes.ne("").sum())),
        incomplete_windows=("complete", lambda done: int((~done).sum())), repeated_item_ids=("duplicates", "sum"))
    return table.rename(index=country_name, level="country")


def month_table(log, country):
    """For one country, per month: items and pages of each collection, retried requests and split windows."""
    mine = log[log["country"] == country]
    table = mine.pivot_table(index="month", columns="collection", values=["items", "pages"], aggfunc="sum", fill_value=0)
    table[("retries", "all")] = mine.groupby("month")["retries"].sum()
    table[("split windows", "all")] = mine.groupby("month")["note"].apply(lambda notes: int(notes.ne("").sum()))
    return table


country, measure, summary = view()
say("Retrieval", "What was requested from the API and how it went. Split windows are automatic smaller requests after an API error or at the item cap.")
display(retrieval_table(RETRIEVAL_LOG))
say(f"Retrieval per month: {country_name(country)}", "Items and pages of each collection, retried requests and split windows. Hazards and exposure are only requested for months with events.")
with pd.option_context("display.max_rows", 80):
    display(month_table(RETRIEVAL_LOG, country))
note("The API reports no total, so completeness rests on every window's last page having no continuation link (see step 5).")


country, measure, summary = view()
caveats = SUMMARIES[country][SUMMARIES[country]["caveat"].fillna("") != ""]
say("Events with a caveat", "PDC's own title and its hazard code disagree, or the event is a tsunami bulletin for a warning region (bulletins are hidden elsewhere unless you show them in step 7).")
if caveats.empty:
    note("No event in this view carries a caveat.")
else:
    display(caveats[["title", "hazard_group", "caveat"]].rename(columns={"title": "Event", "hazard_group": "Hazard", "caveat": "Caveat"}).style.hide(axis="index"))

In [ ]:
#@title Functions: correlation_summary
def correlation_summary(tables):
    """How event families were identified, and how every link between events, hazards and exposure items was judged."""
    families = tables["families"]["correlation_status"].value_counts().rename_axis("how the family was identified").reset_index(name="event families") if len(tables["families"]) else pd.DataFrame()
    links = tables["edges"].groupby(["link_method", "status"]).size().rename("links").reset_index()
    return families, links


country, measure, summary = view()
say("How were events linked?", "Families are keyed by the PDC source event ID; the corr_id is used only without one and is labelled as a fallback. Links locate and validate the exact snapshot.")
family_table, link_table = correlation_summary(TABLES[country])
display(family_table.style.hide(axis="index"))
display(link_table.style.hide(axis="index"))
note("A validated link has a matching source event ID, episode and time span. Ambiguous links keep every candidate; none is chosen.")

In [ ]:
#@title Functions: point_keys, point_sharing, geometry_cards
def point_keys(tables):
    """One row per item with a valid point: its family, collection, item ID and its coordinates as text."""
    parts = []
    for name, column in (("events", "event_item_id"), ("hazards", "hazard_item_id"), ("impacts", "impact_item_id")):
        valid = tables[name][tables[name]["valid_geometry"].astype(bool)]
        parts.append(pd.DataFrame({"family_key": valid["family_key"], "collection": name, "item": valid[column],
                                   "point": valid["longitude"].map("{:.8f}".format) + "," + valid["latitude"].map("{:.8f}".format)}))
    return pd.concat(parts).drop_duplicates()


def point_sharing(tables):
    """Distinct points per event family across its event, hazard and exposure items (PDC copies the event point into its hazard and exposure items)."""
    return point_keys(tables).groupby("family_key")["point"].nunique()


def geometry_cards(tables):
    """Headline numbers about what can safely be mapped."""
    events, points = tables["events"], point_sharing(tables)
    valid = events["valid_geometry"].astype(bool)
    kinds = ", ".join(f"{kind} {count}" for kind, count in events["geometry_type"].fillna("missing").value_counts().items()) or "none"
    return [("Event points usable", f"{int(valid.sum()):,}", f"of {len(events):,} event snapshots; geometry types: {kinds}"),
            ("Families with one shared point", f"{int((points == 1).sum()):,}", f"of {len(points):,}; hazard and exposure points copy the event point"),
            ("Invalid or missing points", f"{int((~valid).sum()):,}", "event snapshots that cannot be mapped")]


country, measure, summary = view()
say("What can be mapped?", "Every PDC point is the event/hazard location; it is not an impact footprint, and no footprint is ever invented.")
cards(geometry_cards(TABLES[country]))
note(MAP_NOTE)

## 15. Export

One folder per country is written to `outputs/notebook/<run id>/<country code>/` with the same evidence bundle the dashboard offers:
`events.csv`, `impact_observations.csv`, `correlation_evidence.csv`, `quality_summary.csv`, `event_summary.csv`, an Excel workbook (`pdc_evidence.xlsx`), `pdc_evidence.geojson`,
`report.md` and `report.html` (with three small charts) and a `manifest.json` that lists every file with its checksum.

Every file keeps the original values and units, the record IDs and the provenance. **No token, local path or signed link is written.** At the end the files are re-opened and their row counts and checksums are checked.
The bundle uses the **measure** chosen in step 7 and the **event** chosen in step 12 (if you opened that section).

In [ ]:
#@title Functions: project_root, run_id
# Where and under what name.
EXPOSURE_NOTE = "Values are original PDC exposure estimates, not confirmed humanitarian outcomes."  # same wording as the dashboard
TEMPORAL_NOTE = "Time is the PDC exposure timestamp derived from the item ID; not observed impact-report or ingestion time."


def project_root():
    """The project folder (the one holding this project's pyproject.toml), or the current folder on Colab and anywhere else."""
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        pyproject = folder / "pyproject.toml"
        if pyproject.is_file() and "guard-pdc-explorer" in pyproject.read_text(encoding="utf-8", errors="ignore"):
            return folder
    return here


def run_id():
    """A readable name for this run: when, which countries, which years."""
    years = str(YEARS[0]) if len(YEARS) == 1 else f"{YEARS[0]}-{YEARS[-1]}"
    return f"{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_{'-'.join(COUNTRIES)}_{years}"

In [ ]:
#@title Functions: cloned_point_records, quality_metrics
# What the evidence can and cannot support.
def cloned_point_records(tables):
    """Items whose point is shared with an item of another collection of the same event family (`point_keys` comes from step 14)."""
    return int((point_keys(tables).groupby(["family_key", "point"])["collection"].transform("nunique") >= 2).sum())


def quality_metrics(tables, country):
    """Counts that describe what the evidence can and cannot support, as a (metric, value) table."""
    events, hazards, impacts, edges = tables["events"], tables["hazards"], tables["impacts"], tables["edges"]
    items = [entry["item"] for store in RAW[country].values() for entry in store.values()]
    metrics = {
        "event_families": len(tables["families"]), "event_snapshots": len(events), "hazard_records": len(hazards), "impact_observations": len(impacts),
        "missing_source_ids": sum(not (item.get("properties") or {}).get("monty:src_event_id") for item in items),
        "validated_correlations": int(edges["status"].eq("validated").sum()), "ambiguous_correlations": int(edges["status"].eq("ambiguous").sum()),
        "correlation_conflicts": int(edges["status"].eq("conflict").sum()), "missing_companions": int(edges["status"].eq("missing_companion").sum()),
        "repeated_item_ids": int(REPEATED_ITEM_IDS.get(country, 0)), "payload_conflicts": int(PAYLOAD_CONFLICTS.get(country, 0)),
        "missing_geometry": sum(item.get("geometry") is None for item in items), "missing_units": int(impacts["original_unit"].isna().sum()),
        "missing_snapshot_times": int(events["snapshot_time"].isna().sum()), "missing_impact_snapshot_times": int(impacts["snapshot_time"].isna().sum()),
        "derived_event_snapshot_times": int(events["snapshot_time_method"].eq(SNAPSHOT_METHOD).sum()), "derived_impact_snapshot_times": int(impacts["snapshot_time_method"].eq(SNAPSHOT_METHOD).sum()),
        "invalid_geometries": int(sum((t["geometry_type"].notna() & ~t["valid_geometry"].astype(bool)).sum() for t in (events, hazards, impacts))),
        "cloned_point_records": cloned_point_records(tables), "gender_categories_available": int(impacts["impact_category"].isin(["women", "men"]).any()),
    }
    return pd.DataFrame(sorted(metrics.items()), columns=["metric", "value"])

In [ ]:
#@title Functions: public_tables, export_counts, write_csv
# The tables that are written.
def public_tables(country, tables):
    """The five exported tables, exactly as the files will hold them."""
    return {"events": public_frame(tables["events"]), "impacts": public_frame(tables["impacts"]), "correlations": public_frame(tables["edges"]),
            "quality": public_frame(quality_metrics(tables, country)), "summary": public_frame(tables["summary"].drop(columns=["footprint_url"]))}


def export_counts(tables):
    return {"event_families": len(tables["families"]), "event_snapshots": len(tables["events"]), "hazard_snapshots": len(tables["hazards"]),
            "impact_observations": len(tables["impacts"]), "correlation_evidence": len(tables["edges"])}


CSV_FILES = {"events.csv": "events", "impact_observations.csv": "impacts", "correlation_evidence.csv": "correlations",
             "quality_summary.csv": "quality", "event_summary.csv": "summary"}


def write_csv(folder, public):
    """The tables as CSV (UTF-8 with a byte-order mark, so Excel opens them correctly)."""
    for file_name, key in CSV_FILES.items():
        public[key].to_csv(folder / file_name, index=False, lineterminator="\n", encoding="utf-8-sig")
    return list(CSV_FILES)

In [ ]:
#@title Functions: write_excel
# The workbook has six sheets: the query, the three evidence tables, the quality counts and the definitions.
DEFINITIONS = [
    ("Event family", "One exact PDC source event family (monty:src_event_id)."),
    ("Event snapshot", "One event STAC item; repeated snapshots are retained as evidence."),
    ("Impact observation", "One original impact item and detail position; original value and unit are preserved."),
    ("PDC exposure", EXPOSURE_NOTE), ("Map point", MAP_NOTE), ("Exposure time", TEMPORAL_NOTE),
    ("Present zero", "The source explicitly reported numeric zero; this is not missing data."),
    ("Missing", "No matching observation was retrieved for the requested category."),
    ("Conflicting", "More than one incompatible source value remains visible."),
    ("Peak and latest", "Peak = the largest value among an event's retained snapshots; latest = the most recent. Never summed."),
]


def write_excel(path, sheets):
    """One workbook with one sheet per frame; text starting with '=' is kept as text."""
    workbook = Workbook()
    workbook.remove(workbook.active)
    for name, frame in sheets.items():
        sheet = workbook.create_sheet(name)
        sheet.append(list(frame.columns))
        for row in frame.itertuples(index=False):
            sheet.append(list(row))
        sheet.freeze_panes = "A2"
        for cell in (cell for cells in sheet.iter_rows(min_row=2) for cell in cells):
            if isinstance(cell.value, str) and cell.value.startswith("="):
                cell.data_type = "s"
    workbook.save(path)

In [ ]:
#@title Functions: query_spec, retrieval_facts
def query_spec(country):
    """The same query as the package's validated QuerySpec. Its fingerprint must equal the one computed in step 4."""
    q = QUERIES[country]["query"]
    spec = QuerySpec(analysis_mode=q["analysis_mode"], country_code=q["country_code"], year=q["year"], end_year=q.get("end_year"), months=tuple(q["months"]),
                     hazard_codes=tuple(q["hazard_codes"]), impact_types=tuple(q["impact_types"]), categories=tuple(q["categories"]), source_mode="api_only", refresh_api_cache=True)
    if spec.fingerprint != QUERIES[country]["fingerprint"]:
        raise ValueError("The query fingerprint differs from the package's: the notebook and the dashboard would not describe the same query.")
    return spec


def retrieval_facts(country):
    """What the retrieval of one country looked like, for the manifest and the report."""
    log = RETRIEVAL_LOG[RETRIEVAL_LOG["country"] == country]
    failed = [f"{row.collection} {row.start[:10]} to {row.end[:10]}: {public_text(str(row.failure))}" for row in log[~log["complete"]].itertuples()]
    return {"provider": "api", "retrieved_at": RETRIEVED_AT, "complete": not failed, "pages": int(log["pages"].sum()), "returned_count": int(log["returned"].sum()),
            "unique_count": sum(len(store) for store in RAW[country].values()), "server_count": None, "failed_partitions": failed, "warnings": failed}

In [ ]:
#@title Functions: finite, geojson_features, write_geojson
def finite(value):
    return None if value is None or (isinstance(value, float) and math.isnan(value)) else value


def geojson_features(summary, measure, selected_family):
    """Event-family points with the chosen measure's peak value."""
    features = []
    for row in summary[summary["valid_point"]].itertuples():
        properties = {"layer": "event_family", "family_key": row.family_key, "source_event_id": row.source_event_id, "title": row.title, "hazard": row.hazard_group,
                      "countries": list(row.countries), "provider": "api", "snapshot_count": int(row.snapshot_count), "selected_category": MEASURE_CATEGORY[measure],
                      "selected_value": finite(getattr(row, f"{measure}_peak")), "selected_value_kind": "peak across snapshots", "selected_unit": None,
                      "selected_family": row.family_key == selected_family, "geometry_meaning": MAP_NOTE}
        features.append({"type": "Feature", "geometry": {"type": "Point", "coordinates": [row.longitude, row.latitude]}, "properties": properties})
    return features


def write_geojson(path, summary, measure, selected_family):
    """The event points, plus the selected event's alert-area polygons when they were fetched and validated."""
    features = geojson_features(summary, measure, selected_family)
    chosen = summary[summary["family_key"] == selected_family]
    footprint = ALERT_AREAS.get(chosen["footprint_url"].iloc[0]) if len(chosen) else None
    for area in (footprint.areas if footprint is not None else ()):
        features.append({"type": "Feature", "geometry": area.geometry, "properties": {
            "layer": "validated_footprint", "family_key": selected_family, "created": area.created.isoformat() if area.created is not None else None,
            "geometry_meaning": ALERT_AREA_NOTE}})
    path.write_text(json.dumps({"type": "FeatureCollection", "features": features}, ensure_ascii=False, indent=2), encoding="utf-8")

In [ ]:
#@title Functions: report_lines, write_report
# The report says what was asked, what came back, how complete it is and how to read it.
def report_lines(country, spec, counts, facts, measure, selected_family, summary):
    """The Markdown report of one country."""
    status = summary[f"{measure}_status"].map(STATUS_WORDS).value_counts()
    selected = summary[summary["family_key"] == selected_family]
    months = "all months" if len(spec.months) == 12 else ", ".join(MONTH_NAMES[m - 1] for m in spec.months)
    lines = [f"# PDC evidence report: {country_name(country)}", "", f"Generated: {datetime.now(timezone.utc).isoformat()}  ",
             "Source: IFRC Montandon STAC API (read-only); collections pdc-events, pdc-hazards, pdc-impacts  ",
             f"Retrieved: {facts['retrieved_at']}; complete: {facts['complete']}  ", f"Query fingerprint: `{spec.fingerprint}`", "", "## What was asked", "",
             f"- Country: {country_name(country)}", f"- Period: {spec.period_label} ({months})", f"- Hazard codes: {', '.join(spec.hazard_codes) or 'all hazards'}",
             f"- Measures retrieved: {', '.join(spec.categories)}", "", "## What came back", ""]
    lines += [f"- {key.replace('_', ' ').title()}: {value:,}" for key, value in counts.items()]
    lines += ["", "## Completeness", "", f"- {facts['pages']:,} pages, {facts['unique_count']:,} unique items, {len(facts['failed_partitions'])} window(s) that could not be read.",
              "- The API reports no total; a window counts as complete when its last page has no continuation link.", "",
              f"## {MEASURE_LABELS[measure]}: status of the events", ""]
    lines += [f"- {word}: {count:,}" for word, count in status.items()] or ["- No events"]
    if len(selected):
        row = selected.iloc[0]
        lines += ["", "## Selected event", "", f"- Title: {row['title'] or 'Untitled'}", f"- Source event ID: {row['source_event_id'] or 'unavailable'}",
                  f"- Hazard: {row['hazard_group']}", f"- PDC updates: {int(row['snapshot_count']):,}", f"- Peak {MEASURE_NOUNS[measure]}: {compact(row[f'{measure}_peak'])}"]
    lines += ["", "## How to read this", "", f"- {EXPOSURE_NOTE}", f"- {MAP_NOTE}", f"- {TEMPORAL_NOTE}",
              "- Exposure is never added across events, snapshots, categories or countries; zero, missing and conflicting values stay distinct.", ""]
    return lines


def write_report(folder, country, spec, counts, facts, measure, selected_family, summary):
    """report.md and report.html, with the SVG charts of the package's report helper."""
    charts = report_charts(summary, spec, MEASURE_CATEGORY[measure])
    for name, (_, svg) in charts.items():
        (folder / name).write_text(svg, encoding="utf-8")
    text = "\n".join(report_lines(country, spec, counts, facts, measure, selected_family, summary))
    files = "\n## Included files\n\n" + "".join(f"- `{name}`\n" for name in [*CSV_FILES, "pdc_evidence.xlsx", "pdc_evidence.geojson", "manifest.json", "report.md", "report.html", *charts])
    (folder / "report.md").write_text(text + files + "".join(f"\n### {caption}\n\n![{caption}]({name})\n" for name, (caption, _) in charts.items()), encoding="utf-8")
    figures = "".join(f"<figure>{svg}<figcaption>{escape(caption)}</figcaption></figure>" for caption, svg in charts.values())
    (folder / "report.html").write_text("<!doctype html><meta charset='utf-8'><title>PDC evidence report</title><style>body{font:16px Arial,sans-serif;max-width:960px;margin:2rem auto;line-height:1.5}"
                                        f"pre{{white-space:pre-wrap}}figure{{margin:1.5rem 0}}figure svg{{max-width:100%;height:auto}}</style><pre>{escape(text + files)}</pre>{figures}", encoding="utf-8")
    return ["report.md", "report.html", *charts]

In [ ]:
#@title Functions: write_manifest, export_country
# The manifest records the query and its fingerprint, how the retrieval went, the row counts and the size and SHA-256 of every other file,
# so the bundle can be checked later.
def write_manifest(folder, country, spec, counts, facts, measure, selected_family):
    """manifest.json: query, fingerprint, retrieval facts, counts and a checksum for every other file."""
    files = {path.name: {"bytes": path.stat().st_size, "sha256": hashlib.sha256(path.read_bytes()).hexdigest()} for path in sorted(folder.iterdir()) if path.name != "manifest.json"}
    manifest = {"schema_version": 1, "software_version": __version__, "generated_at": datetime.now(timezone.utc).isoformat(), "interface": "notebook",
                "query": spec.as_dict(), "query_fingerprint": spec.fingerprint,
                "provider_fingerprints": [{"provider": "api", "query_fingerprint": spec.fingerprint, "local_fingerprint": None, "retrieved_at": RETRIEVED_AT}],
                "selection": {"category": MEASURE_CATEGORY[measure], "family_key": selected_family}, "retrieval": facts, "counts": counts, "files": files,
                "notes": [EXPOSURE_NOTE, MAP_NOTE, TEMPORAL_NOTE]}
    (folder / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2, sort_keys=True), encoding="utf-8")


def export_country(folder, country, measure, selected_family):
    """Write the whole bundle of one country into `folder`."""
    folder.mkdir(parents=True, exist_ok=True)
    tables, spec = TABLES[country], query_spec(country)
    public, counts, facts = public_tables(country, tables), export_counts(tables), retrieval_facts(country)
    write_csv(folder, public)
    sheets = {"Query Metadata": pd.DataFrame([(f"query.{key}", json.dumps(value) if isinstance(value, (list, tuple)) else value) for key, value in spec.as_dict().items()]
                                              + [("query.fingerprint", spec.fingerprint), ("retrieval.retrieved_at", RETRIEVED_AT), ("retrieval.complete", facts["complete"])]
                                              + [(f"count.{key}", value) for key, value in counts.items()], columns=["Field", "Value"]),
              "Events": public["events"], "Impact Observations": public["impacts"], "Correlation Evidence": public["correlations"], "Quality Summary": public["quality"],
              "Definitions": pd.DataFrame(DEFINITIONS, columns=["Term", "Definition"])}
    write_excel(folder / "pdc_evidence.xlsx", sheets)
    write_geojson(folder / "pdc_evidence.geojson", tables["summary"], measure, selected_family)
    write_report(folder, country, spec, counts, facts, measure, selected_family, tables["summary"])
    write_manifest(folder, country, spec, counts, facts, measure, selected_family)
    return sorted(path.name for path in folder.iterdir())


country_shown, measure_shown, _ = view()
selected_event = event_picker.value if "event_picker" in globals() else None
RUN_FOLDER = project_root() / "outputs" / "notebook" / run_id()
print(f"Writing to {RUN_FOLDER.relative_to(project_root()) if RUN_FOLDER.is_relative_to(project_root()) else RUN_FOLDER}")
print(f"Measure: {MEASURE_SHORT[measure_shown]}. Selected event: {selected_event or 'none (open step 12 to choose one)'}.")
WRITTEN = {country: export_country(RUN_FOLDER / country, country, measure_shown, selected_event if country == country_shown else None) for country in COUNTRIES}
for country, names in WRITTEN.items():
    print(f"\n{country_name(country)}: {len(names)} files\n  " + ", ".join(names))

In [ ]:
#@title Functions: token_leaks
def token_leaks(folder):
    """Names of the files in `folder` that contain the token. Text files are searched as text, the workbook cell by cell."""
    needle, leaks = config.api_token, []
    for path in sorted(folder.iterdir()):
        if path.suffix == ".xlsx":
            workbook = load_workbook(path, read_only=True)
            text = "\n".join(str(cell.value) for sheet in workbook.worksheets for row in sheet.iter_rows() for cell in row if cell.value is not None)
            workbook.close()
        else:
            text = path.read_text(encoding="utf-8", errors="ignore")
        if needle in text:
            leaks.append(path.name)
    return leaks


for country in COUNTRIES:
    checked = validate_export_bundle(RUN_FOLDER / country)
    leaks = token_leaks(RUN_FOLDER / country)
    print(f"{country_name(country)}: re-opened and reconciled {checked['events']:,} events, {checked['impacts']:,} exposure values, {checked['correlations']:,} links, "
          f"{checked['geojson_features']:,} map features. Token found in {len(leaks)} file(s).")
    if leaks:
        raise RuntimeError(f"The token was found in {leaks}. Delete {RUN_FOLDER} and do not share it.")

In [ ]:
#@title Functions: zip_run, download_link
# The folders stay on the machine running the notebook (on Binder and Colab that is a temporary machine, so download what you need before
# closing it).
def zip_run(folder):
    """One zip file next to the run folder holding every country folder."""
    return Path(shutil.make_archive(str(folder), "zip", root_dir=folder.parent, base_dir=folder.name))


def download_link(path, limit=40_000_000):
    """A link that downloads the file from the page itself (a small file only; larger files stay in the file browser)."""
    if path.stat().st_size > limit:
        return f"<p>The zip file is large ({path.stat().st_size / 1e6:.0f} MB). Download it from the file browser: <code>{escape(path.name)}</code></p>"
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"<p><a download='{escape(path.name)}' href='data:application/zip;base64,{data}'><b>Download {escape(path.name)}</b></a> ({path.stat().st_size / 1e6:.1f} MB)</p>"


archive = zip_run(RUN_FOLDER)
print(f"Zip file: {archive.name} ({archive.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files

    files.download(str(archive))
else:
    display(HTML(download_link(archive)))

## 16. Evidence rules and limitations

What the numbers in this notebook do and do not mean.

* **Exposure, not outcome.** PDC `affected_total` values are PDC population or asset exposure estimates. They are not deaths, injuries, displacement or confirmed humanitarian outcomes.
* **Original values are preserved.** Values, units (including `None`), categories, types, estimate types, identifiers, geometry, dates, links, processing versions and provenance are kept exactly as published. No currency is assumed for capital exposure, no unit is converted, and a missing value is never replaced.
* **Zero, missing, unavailable and conflicting stay apart.** A zero was reported as 0. *Not published* means no item exists. *Conflicting* means two different values for one snapshot time; neither is chosen.
* **Nothing is added.** Exposure is not summed across snapshots, events, categories or countries. `people` is an all-population value and is never added to the age bands; the age bands' sum is only a consistency check.
* **Event families.** `monty:src_event_id` identifies an event family. One family has several snapshots, and a repeated source ID is not a duplicate. A repeated *item* ID would be. The unchanged `corr_id` is used only when the source event ID is absent, and is labelled `legacy_corr_id_fallback`.
* **Links locate and validate.** `rel=related` links find the exact snapshot; they are checked for source event ID, episode and time. Ambiguous matches are kept as ambiguous. Correlation IDs alone never tie different sources together.
* **Snapshot time** comes from the PDC item ID for known processing versions. It is the time of the exposure estimate, not of an observed impact report and not of ingestion into Montandon.
* **A map point is the event or hazard location.** PDC copies it into its hazard and exposure items. It is not an impact footprint. The alert-area outline is the area PDC used for its alert; no footprint is ever invented, and an unreadable file is reported.
* **Completeness.** The API reports no total number of results. A window counts as complete only when its last page has no continuation link; failed windows are listed, never hidden. Nothing is sampled or truncated.
* **Coverage changes over time.** Montandon holds PDC events only for some years, and differently in each. A change between years can reflect coverage, not hazard activity.
* **This is evidence preparation.** The notebook does not model population exposure and does not forecast.